# taskb_adapter_full

Three full-pipeline seeds; encoder/adapters, common lock and paired official-val. See [Task B guide](README_taskb.md) for report evidence, execution order and historical-source limitations.


## 1. Cấu hình và dependencies
Chọn GPU. Chỉ đổi phase/seed/cờ chạy; recipe đã khóa từ nguồn. Giữtorch/CUDA hiện có; bootstrap đồng bộ SDK trước import. Nếu báo module cũ: lưu notebook/log, restart một lần rồi chạy từ đầu. Không có vòng lặp restart/cài lại toàn môi trường.


In [ ]:
CFG = {'preparation_root': '/content/drive/MyDrive/ĐATN/data/processed/benchmark_preparation/FashionIQ/fashioniq_prep_v1_20261103',
 'output_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ',
 'campaign_id': 'conventional_retrieval_adapter_full_v1',
 'local_root': '/content/visfit_tme_adapter_full_experiment',
 'phase': 'preflight',
 'seeds_to_run': [42],
 'seeds': [42, 43, 44],
 'enable_training': False,
 'enable_lock': False,
 'enable_benchmark': False,
 'require_cuda': True,
 'encoder': {'batch_size': 4,
             'epochs': 3,
             'learning_rate': 0.0001,
             'weight_decay': 0.01,
             'gradient_clip': 1.0,
             'retrieval_temperature': 0.07,
             'checkpoint_every_steps': 1024,
             'precision': 'float32',
             'scheduler': None,
             'scaler': None,
             'freeze_embeddings': True},
 'adapter': {'batch_size': 32,
             'max_epochs': 10,
             'patience': 3,
             'learning_rate': 0.0001,
             'weight_decay': 0.01,
             'gradient_clip': 1.0,
             'retrieval_temperature': 0.07,
             'checkpoint_every_steps': 64,
             'width': 128,
             'heads': 4,
             'precision': 'float32',
             'scheduler': None,
             'scaler': None},
 'encode_batch_size': 16,
 'eval_batch_size': 32,
 'cache_shard_size': 128,
 'export_top_k': 50,
 'baseline_tolerance_pp': 0.01,
 'gate_equivalence_atol': 1e-06,
 'native_atol': 3e-05,
 'native_rtol': 3e-05,
 'legacy_conventional_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ/tme_dev_v1_seed42',
 'legacy_adapter_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v2_storagefix'}


In [ ]:
# Chỉ stdlib trước import model; bootstrap fix: dependency-owner-and-loaded-version-v2.
import sys, subprocess, importlib.util, importlib.metadata, json
from pathlib import Path
BOOTSTRAP = {'python':sys.version,'installs':[],'pins':{},'passed':False,'bootstrap_fix':'dependency-owner-and-loaded-version-v2'}
local=Path('/content/visfit_tme_adapter_full_bootstrap');local.mkdir(parents=True,exist_ok=True)
BOOTSTRAP['loaded_before_bootstrap']={k:{'version':getattr(v,'__version__',None),'path':getattr(v,'__file__',None)} for k,v in list(sys.modules.items()) if k in ('torch','transformers','tokenizers','huggingface_hub')}
PINS = {'transformers':'4.57.1','tokenizers':'0.22.1','huggingface-hub':'0.36.0'}
for package in ('torch','numpy','pandas','Pillow'):
    try: BOOTSTRAP['pins'][package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError('Colab base package missing: '+package+'; use a standard GPU runtime') from None
needed=[]
if importlib.util.find_spec('psutil') is None:needed.append('psutil')
for package,version in PINS.items():
    try: installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version:needed.append(package+'=='+version)
    module={'huggingface-hub':'huggingface_hub'}.get(package,package)
    loaded=any(n==module or n.startswith(module+'.') for n in sys.modules)
    loaded_version=getattr(sys.modules.get(module),'__version__',None)
    if loaded and (installed!=version or loaded_version!=version):
        BOOTSTRAP.update(restart_required=True,blocked_package=package,installed_version=installed,loaded_version=loaded_version)
        (local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
        raise RuntimeError('A dependency requiring a version change is already loaded: '+package+'. Save the notebook and use Runtime > Restart session, then run from the top. No automatic install/restart loop.')
    BOOTSTRAP['pins'][package]=version
if needed:
    p=subprocess.run([sys.executable,'-m','pip','install','--quiet',*needed],capture_output=True,text=True)
    if p.returncode:raise RuntimeError('Dependency installation failed; stopped before importing models. No torch/CUDA upgrade requested.')
    BOOTSTRAP['installs']=needed
check=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
BOOTSTRAP['pip_check']={'returncode':check.returncode,'report':check.stdout+check.stderr}
# Scope by dependent package owner, not a substring of the dependency mentioned in the error.
# Gradio/Diffusers are not used by this pilot; their conflicts remain visible in the report.
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
pending=['torch','torchvision','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow','safetensors','packaging']
core=set()
while pending:
    owner=canonicalize_name(pending.pop())
    if owner in core:continue
    core.add(owner)
    for raw in importlib.metadata.requires(owner) or []:
        req=Requirement(raw)
        if req.marker is None or req.marker.evaluate({'extra':''}):pending.append(req.name)
lines=[line for line in (check.stdout+check.stderr).splitlines() if line.strip() and 'No broken requirements found' not in line]
critical=[line for line in lines if canonicalize_name(line.split()[0]) in core]
BOOTSTRAP['dependency_scope']={'core_packages':sorted(core),'critical_conflicts':critical,'outside_pilot_conflicts':[line for line in lines if line not in critical]}
local=Path('/content/visfit_tme_adapter_full_bootstrap');local.mkdir(parents=True,exist_ok=True)
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if check.returncode and critical:raise RuntimeError('pip check found TME dependency conflicts: '+'; '.join(critical))
probe=subprocess.run([sys.executable,'-c',
    'import torch,transformers; from transformers import CLIPModel,CLIPTokenizer,CLIPImageProcessor; from transformers.generation import GenerationMixin; print(transformers.__version__); print(transformers.__file__)'],capture_output=True,text=True)
BOOTSTRAP['import_probe']={'returncode':probe.returncode,'stdout':probe.stdout,'traceback':probe.stderr}
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if probe.returncode:raise RuntimeError('CLIP dependency import failed in a clean subprocess. Full traceback: /content/visfit_tme_adapter_full_bootstrap/bootstrap_report.json. Diagnose before dataset/training.')
BOOTSTRAP['passed']=True
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
print('Bootstrap đạt; giữ nguyên torch/CUDA:',BOOTSTRAP['pins']['torch'])
print('TME dependency conflicts:',len(critical),'| clean CLIP import:',probe.returncode==0,'| packages installed:',BOOTSTRAP['installs'])
if check.returncode:print('pip check có xung đột ngoài dependency graph của pilot; giữ nguyên đầy đủ trong bootstrap_report.json. Không coi toàn môi trường là pip-clean.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Code tự chứa và hợp đồng dữ liệu
Preparation là nguồn annotation/manifest và feature ảnh đã kiểm chứng; không dùng text feature pretrained thay text của encoder đã train. Chỉ đọc final vision tokens/pooled, không cache12vision layers. Metadata/checksum/ID/mask/EOS/finite được xác minh trước hot path; mapping theo ID. Khi thiếu/hỏng thì dừng, không xóa query hoặc thay zero.


In [ ]:
PREPARATION_ANCHORS = {'data_audit.json': 'fbaa7eae7b99c5e5aeeddcb0bc6027d3870e8c4f9be5658559caa95662321e95', 'feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'extraction_contract.json': 'bbd53b94172aa5c07a358a0697163c620a823b297c6fec6209167bdb59d1f0a1', 'training_plan.json': 'e5496feae33ab0e9dc832e29a1a9097b2d7fed6c2683bd9d4314d6e4973bdc21', 'development_gallery.csv': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'development_val.csv': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'development_train.csv': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'feature_views/development_gallery.csv': '283199425c5fc88fb8e063ea28be629e611e44513689e97f22ee3d0ec5a33927', 'feature_views/development_val.csv': 'cbd8d7b72fa7fb4e3c38d64f3bb8ceeeba73be946e9918c5a77c856a42217619', 'feature_views/development_train.csv': '6f7a5e6ab8792dd8fd08d6fea8c472a9d9b56b59af38ba146467d797f8a98ff0', 'official_train_queries.csv': '539c53d61e9b3d01bf010d089ffa16cef2cd437229feee4a475884586188fd11', 'official_val_queries.csv': '79ea1887f7d5cfd85174754833e4fe2bb6bb74c61c5dd49563cb03b58d9b327c', 'official_val_gallery.csv': '58e274df22b52ed5eb3dbc3428ad2966f56b2fb6202cabc0c194f53e2bb8a3d9'}
EXTRACTION_CONTRACT = {'implementation': 'fashioniq-preparation-v1.0.0', 'encoder_id': 'patrickjohncyh/fashion-clip', 'revision': '7e3ba62ce16b379a1ab479346b66f192e76f51b7', 'encoder_weight_sha256': '7b2e2766a4372167f9db95876b897dcecfe1ca5963f2bf620b3a8393f3ad6366', 'processor': {'_processor_class': None, 'do_resize': True, 'size': {'shortest_edge': 224}, 'resample': 3, 'do_center_crop': True, 'crop_size': {'height': 224, 'width': 224}, 'do_rescale': True, 'rescale_factor': 0.00392156862745098, 'do_normalize': True, 'image_mean': [0.48145466, 0.4578275, 0.40821073], 'image_std': [0.26862954, 0.26130258, 0.27577711], 'do_convert_rgb': True, 'image_processor_type': 'CLIPImageProcessor'}, 'tokenizer': {'class': 'CLIPTokenizer', 'vocab_sha256': 'f9190aead40a965a09db1ba2e556dccac0eda8423cb4926bcd994722899ec68e', 'special_tokens': {'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, 'max_length': 77}, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'truncation_policy': 'error_if_over_77_preserve_all_source_rows', 'dtype': 'float32', 'attention_backend': 'eager', 'autocast': False, 'tf32': False, 'libraries': {'torch': '2.11.0+cu130', 'transformers': '4.57.1', 'tokenizers': '0.22.1', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'Pillow': '11.3.0'}, 'device': 'cuda', 'device_name': 'Tesla T4', 'cuda': '13.0', 'batch_size': 32, 'feature_shard_size': 256, 'reference': 'vision_model.last_hidden_state INCLUDING CLS; no projection or L2', 'text': 'text_model.last_hidden_state, [77,512], no projection/normalization + binary attention_mask', 'target': 'visual_projection(vision_model.pooler_output) then L2, [512]', 'repeat_acceptance': {'atol': 6.103515625e-05, 'rtol': 6.103515625e-05, 'predeclared': True, 'note': 'Repeat same input and execution contract; not a historical feature acceptance threshold.'}}
LEGACY = {'adapter_config_sha256': 'e8ccdc60d5e7c655a3a9009bb8b093a4e95d2d0074d18442a6ed3d77da5dfe5d', 'adapter_code_sha256': '63c5666a61a6fb4afbee2b84f49dbc3d77fa8a2ee14bc47e91f34afd23674dd2', 'adapter_best': {'phase': 'retrieval_only_development', 'sha256': '7d92ef45dfa5cec81ad9042b6bc1eceb2abfdfc7cc957595083d8d2c3c923783', 'signature': '0d0fd5fbacb1bb940499a2046e1c18a489792f38dce6406134c3de6f0df0e0d9', 'step': 5070}, 'adapter_last': {'phase': 'retrieval_only_development', 'sha256': '7d92ef45dfa5cec81ad9042b6bc1eceb2abfdfc7cc957595083d8d2c3c923783', 'signature': '0d0fd5fbacb1bb940499a2046e1c18a489792f38dce6406134c3de6f0df0e0d9', 'step': 5070}, 'encoder_best': {'phase': 'conventional', 'sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'signature': 'c6cf0642cdcf5b7c7c5fc8fe6246fd7f0c668377036fefb12d45cfb1c10dc3d3', 'step': 12153}, 'encoder_config_sha256': 'b3d48c8550a8f36ee1bbbc575878d10ac10e416509b8a599ba23019c7561935f', 'encoder_metrics': {'Mean': 72.6305734380787, 'gallery_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'n_query': 1796, 'overall': {'R@1': 25.833096037059164, 'R@10': 61.020876488685154, 'R@5': 49.44016690768524, 'R@50': 84.24027038747225}, 'per_category': {'dress': {'R@1': 22.073578595317723, 'R@10': 54.515050167224075, 'R@5': 43.97993311036789, 'R@50': 79.76588628762542}, 'shirt': {'R@1': 27.545909849749584, 'R@10': 61.769616026711184, 'R@5': 51.41903171953256, 'R@50': 84.47412353923205}, 'toptee': {'R@1': 27.879799666110184, 'R@10': 66.7779632721202, 'R@5': 52.92153589315526, 'R@50': 88.48080133555926}}, 'query_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'scope': 'full locked development validation; NOT official-val', 'search': 'exact L2 cosine FP32, reference retained, exact ties original gallery_index ascending', 'unit': 'percent'}, 'encoder_text_hash': 'acc814a8bb4c419c3a7489bc3efbf000564d5e8eae3660dfa42ceb74b332d265', 'adapter_initial_sha256': 'd6df8bce9821c85b8c57e8b235954bdde3bfc833fb6ec064a01e145be60afb1e'}
SCHEMA = "conventional-retrieval-adapter-full-v1"
CAPTION_POLICY = "clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1"
CATEGORIES = ("dress","shirt","toptee")

CODE_SHA256 = '99c429ec11344e84e42ddc48032cec01a0aef14f9da98408796f77e85f2c7083'
SCIENCE_SHA256 = 'b1fd1d4e20dc3792f97c290a87890a876eaab981d69b9ad25e42081f234f2b31'


In [ ]:
# common.py — embedded runtime
import os,json,hashlib,math,random,time,copy,traceback,uuid,shutil,platform,tempfile,contextlib,importlib.metadata
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from transformers import CLIPModel,CLIPTokenizer,CLIPImageProcessor

def require(condition, message):
    if not condition: raise RuntimeError(message)

def canonical(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()

def fingerprint(x): return hashlib.sha256(canonical(x)).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text())

def write_bytes(path, raw, immutable=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if immutable and path.exists():
        require(path.read_bytes() == raw, 'Existing immutable artifact differs: '+str(path)); return
    tmp = path.with_name(path.name+'.'+uuid.uuid4().hex+'.partial')
    tmp.write_bytes(raw); os.replace(tmp, path)

def write_json(path, value, immutable=False):
    write_bytes(path, canonical(value)+b'\n', immutable)

def write_csv(path, frame, immutable=False):
    write_bytes(path, frame.to_csv(index=False, lineterminator='\n').encode(), immutable)

def safe_child(root, relative):
    root = Path(root).resolve(); path = (root/relative).resolve()
    require(path != root and root in path.parents, 'Path escapes root'); return path

def tensor_hash(module):
    h = hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        h.update(name.encode()); h.update(str(value.dtype).encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def finite(x, name):
    require(bool(torch.isfinite(x).all()), 'NaN/Inf: '+name); return x

def unit(x):
    finite(x, 'unnormalized vector')
    require(bool((x.float().norm(dim=-1)>1e-8).all()), 'Zero retrieval vector')
    return F.normalize(x.float(), dim=-1)

def normalize_caption(x): return x.strip('.?, ')

def caption_text(a, b): return normalize_caption(a).capitalize()+' and '+normalize_caption(b)

def assess_captions(caps):
    if not isinstance(caps, list) or len(caps)!=2 or not all(isinstance(x,str) for x in caps):
        return 'schema_error', None
    content = [bool(normalize_caption(x)) and not normalize_caption(x).isspace() for x in caps]
    if not any(content): return 'both_empty_captions', None
    return None, 'single_empty_caption' if not all(content) else None

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'])
    if s['cuda'] is not None:
        require(torch.cuda.is_available(), 'Resume CUDA checkpoint on CPU refused')
        torch.cuda.set_rng_state_all(s['cuda'])

def epoch_batches(frame,batch,seed,epoch):
    ids=np.asarray(sorted(frame.query_id),dtype=object);order=np.random.default_rng(seed+epoch).permutation(ids)
    require(len(ids)>=2 and batch>=2,'Contrastive batch too small')
    batches=[order[i:i+batch].tolist() for i in range(0,len(ids),batch)]
    if len(batches)>1 and len(batches[-1])==1:batches[-2].extend(batches.pop())
    return batches

def exact_ranks(query,gallery,target_positions):
    q=np.asarray(query,dtype=np.float32); g=np.asarray(gallery,dtype=np.float32)
    require(q.ndim==g.ndim==2 and q.shape[1]==g.shape[1] and len(q)>0 and len(g)>0,'Ranking shapes invalid')
    require(np.isfinite(q).all() and np.isfinite(g).all(),'Nonfinite ranking input')
    require((np.linalg.norm(q,axis=-1)>1e-8).all() and (np.linalg.norm(g,axis=-1)>1e-8).all(),'Zero ranking input')
    q=q/np.linalg.norm(q,axis=-1,keepdims=True);g=g/np.linalg.norm(g,axis=-1,keepdims=True)
    targets=np.asarray(target_positions,dtype=np.int64)
    require(targets.shape==(len(q),) and (targets>=0).all() and (targets<len(g)).all(),'Target gallery alignment invalid')
    scores=q@g.T; ts=scores[np.arange(len(q)),targets]
    ranks=1+(scores>ts[:,None]).sum(1)+((scores==ts[:,None])&(np.arange(len(g))[None,:]<targets[:,None])).sum(1)
    return ranks

def macro_metrics(per_category):
    require(set(per_category)==set(CATEGORIES),'Macro requires three categories')
    overall={f'R@{k}':float(np.mean([per_category[c][f'R@{k}'] for c in CATEGORIES])) for k in (1,5,10,50)}
    return {'overall':overall,'Mean':(overall['R@10']+overall['R@50'])/2}

@contextlib.contextmanager
def isolated_rng():
    saved=rng_state()
    try:yield
    finally:restore_rng(saved)

def same_rng(a,b):
    cpu=a['python']==b['python'] and np.array_equal(a['numpy'][1],b['numpy'][1]) and a['numpy'][2:]==b['numpy'][2:] and torch.equal(a['torch'],b['torch'])
    cuda=(a['cuda'] is None and b['cuda'] is None) or (a['cuda'] is not None and b['cuda'] is not None and len(a['cuda'])==len(b['cuda']) and all(torch.equal(x,y) for x,y in zip(a['cuda'],b['cuda'])))
    return cpu and cuda

def cpu_tree(x):
    if torch.is_tensor(x):return x.detach().cpu().clone()
    if isinstance(x,dict):return {k:cpu_tree(v) for k,v in x.items()}
    if isinstance(x,list):return [cpu_tree(v) for v in x]
    if isinstance(x,tuple):return tuple(cpu_tree(v) for v in x)
    return copy.deepcopy(x)

def frame_sha(frame):return hashlib.sha256(frame.to_csv(index=False,lineterminator='\n').encode()).hexdigest()
def csv_frame(path):return pd.read_csv(path,dtype=str,keep_default_na=False)
def stamp():return pd.Timestamp.now(tz='UTC').isoformat()

def physical_runtime():
    return {'packages':{k:importlib.metadata.version(k) for k in ('torch','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow')},'python':platform.python_version(),'cuda':torch.version.cuda,'device':str(DEVICE),'gpu':torch.cuda.get_device_name() if DEVICE.type=='cuda' else None,'dtype':'float32','attention':'eager','amp':False,'tf32':False}

@contextlib.contextmanager
def measured(folder,label):
    started=time.perf_counter();outcome='passed'
    try:yield
    except BaseException:
        outcome='failed';write_bytes(Path(folder)/(label+'_traceback.txt'),traceback.format_exc().encode());raise
    finally:
        path=Path(folder)/'phase_timing.json';rows=read_json(path) if path.exists() else []
        rows.append({'stage':label,'elapsed_seconds':time.perf_counter()-started,'status':outcome,'utc':stamp(),'scope':'Observed inclusive stage wall time; do not add nested compute/validation/checkpoint counters again'})
        write_json(path,rows)

def encoder_recipe():return {**CFG['encoder'],'optimizer':'AdamW','betas':[.9,.999],'eps':1e-8,'vision_frozen':True,'trainable':'text encoder blocks + final LN + text projection; embeddings frozen','objective':'symmetric multi-positive InfoNCE by target ID','composition':'unit(unit(project(EOS)) + unit(reference pooled))'}
def adapter_recipe():return {**CFG['adapter'],'optimizer':'AdamW','betas':[.9,.999],'eps':1e-8,'encoder_frozen':True,'projection_frozen_buffer':True,'trainable_parameters':446978,'objective':'symmetric multi-positive InfoNCE by target ID','alignment_loss':False,'gates':'zero init, tanh RI/TA, source residual stream'}

def init_campaign():
    global ROOT,LOCAL,DEVICE,SESSION_ID,_PREFLIGHT_RECEIPT
    require(CFG['seeds']==[42,43,44] and set(CFG['seeds_to_run'])<=set(CFG['seeds']) and len(set(CFG['seeds_to_run']))==len(CFG['seeds_to_run']) and CFG['seeds_to_run'],'Full-pipeline seeds must be 42,43,44; selection must be nonempty unique subset')
    require(CFG['encoder']['epochs']==3 and CFG['encoder']['batch_size']==4 and CFG['adapter']['batch_size']==32 and CFG['adapter']['width']==128 and CFG['adapter']['heads']==4,'Preserve verified budgets/batches/architecture; new recipe requires explicit reviewed campaign')
    require(CFG['encoder']['precision']==CFG['adapter']['precision']=='float32','FP32 locked')
    require(Path(CFG['campaign_id']).name==CFG['campaign_id'],'Invalid campaign ID')
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu');require(not CFG['require_cuda'] or DEVICE.type=='cuda','Choose Colab GPU before preflight')
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False;torch.backends.cudnn.benchmark=False;torch.set_default_dtype(torch.float32)
    require(not torch.is_autocast_enabled(),'Disable an inherited autocast context; the experiment requires FP32')
    ROOT=Path(CFG['output_root'])/CFG['campaign_id'];LOCAL=Path(CFG['local_root'])/CFG['campaign_id'];SESSION_ID=uuid.uuid4().hex;_PREFLIGHT_RECEIPT=None
    for original in (CFG['preparation_root'],CFG['legacy_conventional_root'],CFG['legacy_adapter_root']):
        p=Path(original).resolve();r=ROOT.resolve();require(r!=p and p not in r.parents and r not in p.parents,'Campaign output overlaps protected source')
    ROOT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
    binding={'schema':SCHEMA,'seeds':CFG['seeds'],'encoder':encoder_recipe(),'adapter':adapter_recipe(),'preparation_anchors':PREPARATION_ANCHORS,'science_sha256':SCIENCE_SHA256,'caption_policy':CAPTION_POLICY,'seed_scope':'full pipeline: encoder seed == adapter seed'}
    write_json(ROOT/'campaign_contract.json',binding,immutable=True);write_json(ROOT/'latest_runtime.json',physical_runtime())
    write_json(ROOT/'bootstrap_report.json',BOOTSTRAP)


In [ ]:
# models.py — embedded runtime
def eos_positions(ids, mask, eos_token_id):
    require(ids.shape==mask.shape and bool(((mask==0)|(mask==1)).all()), 'Token/mask shape or values invalid')
    lengths=mask.sum(-1)
    require(bool((lengths>=2).all()), 'Empty text token sequence')
    expected=torch.arange(ids.shape[1],device=ids.device)[None,:]<lengths[:,None]
    require(torch.equal(expected,mask.bool()), 'Tokenizer must right-pad contiguous sequences')
    # pad_token == EOS for CLIP; intersect with attention_mask and select first real EOS.
    is_eos=(ids==eos_token_id)&mask.bool()
    require(bool(is_eos.any(-1).all()), 'EOS absent from unpadded tokens')
    pos=is_eos.long().argmax(-1)
    require(bool((pos==lengths-1).all()), 'EOS must close each untruncated modifier')
    return pos

def text_masks(mask, dtype):
    b,n=mask.shape; neg=torch.finfo(dtype).min
    padding=torch.zeros((b,1,n,n),device=mask.device,dtype=dtype).masked_fill(~mask[:,None,None,:].bool(),neg)
    causal=torch.full((n,n),neg,dtype=dtype,device=mask.device).triu(1)
    return padding,causal[None,None].expand(b,1,n,n)

class ResidualAdapter(nn.Module):
    def __init__(self,projection_weight,text_dim=512,vision_dim=768,width=128,heads=4):
        super().__init__();require(width%heads==0,'Adapter width/head mismatch')
        self.register_buffer('projection_weight',projection_weight.detach().clone())
        self.text_norm=nn.LayerNorm(text_dim);self.text_down=nn.Linear(text_dim,width)
        self.reference_norm=nn.LayerNorm(vision_dim);self.reference_down=nn.Linear(vision_dim,width)
        self.cross_attention=nn.MultiheadAttention(width,heads,dropout=0.,batch_first=True)
        self.ri_mlp=nn.Sequential(nn.Linear(width,width),nn.GELU(),nn.Linear(width,text_dim))
        self.ta_mlp=nn.Sequential(nn.Linear(text_dim,width),nn.GELU(),nn.Linear(width,text_dim))
        self.gate_ri=nn.Parameter(torch.zeros(()));self.gate_ta=nn.Parameter(torch.zeros(()))
    def forward(self,reference_features,text_features,mask,eos,return_aux=False,gates_off=False):
        require(text_features.ndim==3 and mask.shape==text_features.shape[:2] and eos.shape==(len(text_features),),'Adapter input shape')
        require(bool(((mask==0)|(mask==1)).all()) and bool((eos==mask.sum(-1)-1).all()),'Mask/EOS invalid')
        require(bool((eos>=1).all()) and bool((mask.sum(-1)<=mask.shape[1]).all()),'Invalid sequence length')
        require(torch.equal(mask.bool(),torch.arange(mask.shape[1],device=mask.device)[None,:]<mask.sum(-1)[:,None]),'Mask must be contiguous right padding')
        valid=mask.unsqueeze(-1).to(text_features.dtype)
        x=self.text_norm(text_features);q=self.text_down(x);kv=self.reference_down(self.reference_norm(reference_features['tokens']))
        ri=self.cross_attention(q,kv,kv,need_weights=False)[0];ri=self.ri_mlp(ri)*valid
        ta=self.ta_mlp(x)*valid
        gri=self.gate_ri.new_zeros(()) if gates_off else torch.tanh(self.gate_ri)
        gta=self.gate_ta.new_zeros(()) if gates_off else torch.tanh(self.gate_ta)
        enhanced=(text_features+gri*ri+gta*ta)*valid # no LN/MLP on the base stream
        index=torch.arange(len(eos),device=eos.device)
        text=unit(F.linear(enhanced[index,eos],self.projection_weight))
        query=unit(text+unit(reference_features['pooled'])) # reference composed exactly once
        if return_aux:
            student=F.linear((text_features+ta)[index,eos],self.projection_weight)
            return query,student,enhanced
        return query
    def encode_query(self,reference_features,text_features,mask,eos):
        return self(reference_features,text_features,mask,eos)
    def enhanced_text_sequence(self,reference_features,text_features,mask,eos):
        _,_,h=self(reference_features,text_features,mask,eos,return_aux=True)
        return h,mask,eos

def positive_mask(target_ids,device):
    require(target_ids and all(isinstance(v,str) and v for v in target_ids),'Invalid target IDs')
    return torch.tensor([[a==b for b in target_ids] for a in target_ids],dtype=torch.bool,device=device)

def retrieval_loss(query,target,target_ids,temperature):
    require(temperature>0 and query.shape==target.shape,'InfoNCE shape/temperature')
    logits=unit(query)@unit(target.detach()).T/temperature;p=positive_mask(target_ids,query.device)
    def direction(x,m):return -((F.log_softmax(x,dim=1)*m).sum(1)/m.sum(1)).mean()
    return (direction(logits,p)+direction(logits.T,p.T))/2

class ConventionalOnly(nn.Module):
    """Exactly the source conventional path; no RI/TA/full-depth modules are created."""
    def __init__(self,text,projection,eos_id):
        super().__init__();self.text=copy.deepcopy(text);self.projection=copy.deepcopy(projection);self.eos_id=int(eos_id)
        self.requires_grad_(True);self.text.embeddings.requires_grad_(False)
    def encode_text(self,modifier):
        ids,mask=modifier['input_ids'],modifier['attention_mask'];eos=eos_positions(ids,mask,self.eos_id)
        h=self.text.embeddings(input_ids=ids);padding,causal=text_masks(mask,h.dtype);valid=mask.unsqueeze(-1).to(h.dtype)
        for block in self.text.encoder.layers:h=block(h,attention_mask=padding,causal_attention_mask=causal)[0]*valid
        return self.text.final_layer_norm(h)*valid,mask,eos
    def encode_query(self,reference,modifier):
        h,mask,eos=self.encode_text(modifier);text=self.projection(h[torch.arange(len(h),device=h.device),eos])
        return unit(unit(text)+unit(reference['pooled']))
    def freeze(self):self.requires_grad_(False);self.eval();return self

class CachedConventional(nn.Module):
    def __init__(self,projection):super().__init__();self.register_buffer('projection_weight',projection.detach().clone())
    def forward(self,ref,text,mask,eos):
        return unit(unit(F.linear(text[torch.arange(len(eos),device=eos.device),eos],self.projection_weight))+unit(ref['pooled']))
    def encode_query(self,*args):return self(*args)

def load_pretrained(package):
    c=package.contract
    with isolated_rng():
        base=CLIPModel.from_pretrained(c['encoder_id'],revision=c['revision'],dtype=torch.float32,attn_implementation='eager')
        require(tensor_hash(base)==c['encoder_weight_sha256'],'Pinned pretrained weights differ')
        tokenizer=CLIPTokenizer.from_pretrained(c['encoder_id'],revision=c['revision']);processor=CLIPImageProcessor.from_pretrained(c['encoder_id'],revision=c['revision'])
        require(fingerprint(tokenizer.get_vocab())==c['tokenizer']['vocab_sha256'] and tokenizer.special_tokens_map==c['tokenizer']['special_tokens'] and tokenizer.padding_side=='right','Tokenizer contract differs')
        require(all(processor.to_dict().get(k)==v for k,v in c['processor'].items() if k not in ('_processor_class','image_processor_type')),'Preprocessing contract differs')
        base.float().to(DEVICE).requires_grad_(False).eval()
    return base,tokenizer,processor

def fresh_encoder(base,tokenizer,seed):
    with isolated_rng():
        seed_all(seed);model=ConventionalOnly(base.text_model,base.text_projection,tokenizer.eos_token_id).float().to(DEVICE)
    return model

def fresh_adapter(encoder,seed):
    with isolated_rng():
        seed_all(seed);model=ResidualAdapter(encoder.projection.weight,width=CFG['adapter']['width'],heads=CFG['adapter']['heads']).float().to(DEVICE)
    require(sum(p.numel() for p in model.parameters() if p.requires_grad)==446978,'Residual adapter architecture/parameter count differs')
    return model

def immutable_backbone(base):
    require(not base.training and all(not p.requires_grad and p.grad is None for p in base.parameters()),'Pretrained/vision backbone changed mode/grad')
    return {'vision':tensor_hash(base.vision_model),'visual_projection':tensor_hash(base.visual_projection)}


In [ ]:
# stores.py — embedded runtime
class PreparationStore:
    def __init__(self,source,kind):
        require(kind in ('reference','target'),'Only final image features may be reused')
        self.kind=kind;self.root=source['prep']/'features'/source['meta']['fingerprint']/kind
        require(file_hash(self.root/'index.json')==source['meta']['store_index_sha256'][kind],'Source feature index SHA')
        self.index=read_json(self.root/'index.json');self.ids=self.index['ids'];self.mapping={v:i for i,v in enumerate(self.ids)}
        require(len(self.mapping)==len(self.ids) and self.index['fingerprint']==source['meta']['fingerprint'] and self.index['kind']==kind,'Source index identity')
        mapping_path=self.root.parent/source['meta']['mapping_files'][kind]
        require(file_hash(mapping_path)==source['mapping_hashes'][kind] and read_json(mapping_path)==self.mapping,'Source mapping differs from conventional')
        require(self.index['shape_tail']==({'reference':[50,768],'target':[512]}[kind]) and self.index['dtype']=='float32','Image feature schema')
        self.local=LOCAL/'source_images'/source['meta']['fingerprint']/kind;self.local.mkdir(parents=True,exist_ok=True)
        self.verified=set();self.loaded=OrderedDict()
    def take(self,ids):
        result=[];step=self.index['shard_size']
        for image_id in ids:
            require(image_id in self.mapping,'Image ID absent: '+image_id)
            row=self.mapping[image_id];start=row//step*step;end=min(start+step,len(self.ids));name=f'{start:07d}_{end:07d}'
            local=self.local/(name+'.npy')
            if start not in self.verified:
                marker=read_json(self.root/(name+'.json'))
                expected={'fingerprint':self.index['fingerprint'],'kind':self.kind,'ids_sha256':fingerprint(self.ids[start:end]),'shape':[end-start,*self.index['shape_tail']],'dtype':'float32'}
                require(all(marker.get(k)==v for k,v in expected.items()),'Source shard schema/ID mismatch')
                if not local.is_file() or file_hash(local)!=marker['sha256']:
                    temp=local.with_suffix('.copying');shutil.copyfile(self.root/(name+'.npy'),temp)
                    require(file_hash(temp)==marker['sha256'],'Source feature corrupt');os.replace(temp,local)
                a=np.load(local,mmap_mode='r',allow_pickle=False)
                require(list(a.shape)==expected['shape'] and a.dtype==np.float32 and np.isfinite(a).all(),'Source contents invalid')
                if self.kind=='target':require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Source target not unit')
                self.verified.add(start)
            if start not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[start]=np.load(local,mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(start);result.append(np.asarray(self.loaded[start][row-start]).copy())
        return np.stack(result)

class TrialStore:
    def __init__(self,root,local,contract,specs,shard_size):
        self.key=fingerprint(contract);self.root=Path(root)/self.key;self.local=Path(local)/self.key
        self.root.mkdir(parents=True,exist_ok=True);self.local.mkdir(parents=True,exist_ok=True)
        self.specs=specs;self.contract=contract;self.step=shard_size;self.maps={k:{v:i for i,v in enumerate(s['ids'])} for k,s in specs.items()}
        require(all(len(self.maps[k])==len(s['ids']) for k,s in specs.items()),'Duplicate cached IDs')
        self.index={'fingerprint':self.key,'contract':contract,'specs':specs,'shard_size':shard_size}
        write_json(self.root/'index.json',self.index,immutable=True);self.loaded=OrderedDict();self.paths={}
    def prepare(self,role,compute):
        spec=self.specs[role];ids=spec['ids'];dtype=np.dtype(spec['dtype']);tail=spec['shape']
        (self.root/role).mkdir(exist_ok=True);(self.local/role).mkdir(exist_ok=True)
        for start in range(0,len(ids),self.step):
            end=min(start+self.step,len(ids));name=f'{start:07d}_{end:07d}';p=self.root/role/(name+'.npy');marker=p.with_suffix('.json');pending=p.with_suffix('.pending.json')
            expected={'fingerprint':self.key,'role':role,'ids_sha256':fingerprint(ids[start:end]),'shape':[end-start,*tail],'dtype':spec['dtype']}
            if not marker.is_file() and p.is_file() and pending.is_file():
                m=read_json(pending);require(all(m.get(k)==v for k,v in expected.items()) and file_hash(p)==m['sha256'],'Pending cache corrupt');write_json(marker,m,immutable=True)
            local=self.local/role/p.name
            if marker.is_file():
                m=read_json(marker);require(all(m.get(k)==v for k,v in expected.items()),'Cache fingerprint/ID/shape differs')
                require(p.is_file() and file_hash(p)==m['sha256'],'Committed Drive cache corrupt')
                if not local.is_file() or file_hash(local)!=m['sha256']:
                    temp=local.with_suffix('.copying');shutil.copyfile(p,temp);require(file_hash(temp)==m['sha256'],'Local cache copy corrupt');os.replace(temp,local)
            else:
                require(not p.exists(),'Uncommitted orphan cache; preserve and investigate')
                a=np.asarray(compute(ids[start:end]),dtype=dtype)
                require(list(a.shape)==expected['shape'] and np.isfinite(a).all(),'Computed cache shape/finite')
                temp=local.with_suffix('.partial');
                with temp.open('wb') as f:np.save(f,a,allow_pickle=False)
                m={**expected,'sha256':file_hash(temp)};write_json(pending,m)
                remote=p.with_suffix('.copying');shutil.copyfile(temp,remote);require(file_hash(remote)==m['sha256'],'Cache sync corrupt')
                os.replace(remote,p);write_json(marker,m,immutable=True);os.replace(temp,local)
            a=np.load(local,mmap_mode='r',allow_pickle=False)
            require(list(a.shape)==expected['shape'] and a.dtype==dtype and np.isfinite(a).all(),'Local cache invalid')
            if role in ('reference_pooled','target_pooled'):require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Cached embedding not unit')
            self.paths[role,start]=local
    def take(self,role,ids):
        # Training hot path: prevalidated local mmap, no Drive/JSON/checksum access.
        result=[]
        for image_id in ids:
            row=self.maps[role][image_id];start=row//self.step*self.step;key=(role,start)
            require(key in self.paths,'Run cache preflight before use')
            if key not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[key]=np.load(self.paths[key],mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(key);result.append(np.asarray(self.loaded[key][row-start]).copy())
        return np.stack(result)

def checkpoint_save(path,state,signature):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    require(state['signature']==signature,'Checkpoint signature differs')
    with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as f:temp=Path(f.name)
    try:
        torch.save(state,temp);meta={'signature':signature,'sha256':file_hash(temp),'phase':state.get('phase'),'step':state.get('step')}
        write_json(path.with_suffix('.pending.json'),meta);os.replace(temp,path);write_json(path.with_suffix('.json'),meta)
    finally:
        if temp.exists():temp.unlink()

def checkpoint_load(path,signature):
    path=Path(path);marker=path.with_suffix('.json');pending=path.with_suffix('.pending.json');sha=file_hash(path)
    if pending.is_file():
        p=read_json(pending)
        if p.get('sha256')==sha:
            require(p['signature']==signature,'Pending checkpoint signature')
            if not marker.is_file() or read_json(marker)!=p:write_json(marker,p)
    m=read_json(marker);require(m['signature']==signature and m['sha256']==sha,'Checkpoint checksum/config mismatch')
    state=torch.load(path,map_location='cpu',weights_only=False)
    require(state['signature']==signature,'Checkpoint payload identity differs');return state

def sync_checkpoint(local,destination,signature):
    destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)
    marker=read_json(Path(local).with_suffix('.json'));require(marker['signature']==signature,'Checkpoint sync signature')
    write_json(destination.with_suffix('.pending.json'),marker)
    temp=destination.with_suffix('.copying');shutil.copyfile(local,temp);require(file_hash(temp)==marker['sha256'],'Drive checkpoint sync corrupt')
    os.replace(temp,destination);write_json(destination.with_suffix('.json'),marker)


def committed_checkpoint_load(path,signature):
    # Protected/complete artifacts are read only. Never repair their pending/commit markers.
    path=Path(path);m=read_json(path.with_suffix('.json'))
    require(m['signature']==signature and m['sha256']==file_hash(path),'Committed source checkpoint checksum/signature mismatch')
    state=torch.load(path,map_location='cpu',weights_only=False)
    require(state['signature']==signature,'Committed source checkpoint payload signature differs');return state


In [ ]:
# data.py — embedded runtime
class Preparation:
    def __init__(self,root):
        self.root=Path(root);self.frames={};self.hashes={};self.views={}
        for name,sha in PREPARATION_ANCHORS.items():
            require(file_hash(safe_child(self.root,name))==sha,'Preparation anchor changed: '+name)
        self.protocol=read_json(self.root/'protocol_config.json');self.contract=read_json(self.root/'extraction_contract.json');self.meta=read_json(self.root/'feature_metadata.json')
        audit=read_json(self.root/'data_audit.json')
        require(file_hash(self.root/'feature_audit.json')==self.meta['feature_audit_sha256'],'Preparation feature audit checksum differs')
        feature_audit=read_json(self.root/'feature_audit.json');require(feature_audit['passed'] and not feature_audit['partial'] and feature_audit['fingerprint']==self.meta['fingerprint'],'Incomplete feature extraction audit')
        require(self.contract==EXTRACTION_CONTRACT and self.meta['contract']==self.contract and self.meta['status']=='complete' and not self.meta['legacy_arrays_mixed'],'Preparation/extraction contract incomplete or differs')
        require(audit['passed'] and not audit['partial_mode'] and audit['schema_error_count']==audit['both_empty_captions_count']==audit['missing_or_corrupt_image_count']==0,'Preparation data/image audit blocked')
        require(self.protocol['caption_policy']==CAPTION_POLICY and self.protocol['reference_policy']=='retain in gallery (FashionIQ)' and self.protocol['actual_counts']==audit['counts'],'Locked annotation/protocol counts differ')
        self.binding={'anchors':PREPARATION_ANCHORS,'feature_fingerprint':self.meta['fingerprint'],'protocol_sha256':file_hash(self.root/'protocol_config.json'),'extraction_sha256':file_hash(self.root/'extraction_contract.json')}
        for name in ('development_train','development_val','development_gallery','official_train_queries'):self.read(name)
        train,val,full=[self.frames[x] for x in ('development_train','development_val','official_train_queries')]
        require(len(train)==16204 and len(val)==1796 and len(full)==18000,'Locked development/final training counts differ')
        require(set(train.query_id).isdisjoint(val.query_id) and set(train.target_image_id).isdisjoint(val.target_image_id),'Development target/query leakage')
        joined=pd.concat([train,val],ignore_index=True).sort_values('query_id').reset_index(drop=True)
        require(joined[full.columns].equals(full.sort_values('query_id').reset_index(drop=True)),'Full official-train != exact union of development partitions by query ID')
    def read(self,name):
        if name in self.frames:return self.frames[name]
        path=safe_child(self.root,name+'.csv');require(file_hash(path)==PREPARATION_ANCHORS[name+'.csv'],'Manifest SHA differs: '+name)
        f=csv_frame(path);view_spec=self.meta['feature_views'][name];view_path=safe_child(self.root,view_spec['path'])
        require(file_hash(view_path)==view_spec['sha256'],'Feature view SHA differs: '+name)
        view=csv_frame(view_path);require(len(f)==view_spec['rows'] and f.equals(view[f.columns]),'Feature view row/ID alignment differs')
        if 'query_id' in f:
            split='val' if name=='official_val_queries' else 'train'
            require(f.query_id.is_unique and set(f.official_split)=={split},'Query ID or official split invalid')
            warnings=[]
            for row in f.itertuples():
                require(row.category in CATEGORIES and row.source_file==f'cap.{row.category}.{split}.json' and row.query_id==f'{row.category}:{split}:{int(row.source_row):06d}','Annotation/source mapping invalid: '+row.query_id)
                for image_id in (row.reference_image_id,row.target_image_id):require(bool(image_id) and image_id==image_id.strip() and '/' not in image_id and '\\' not in image_id,'Invalid image ID')
                error,warning=assess_captions([row.caption_1,row.caption_2]);require(error is None,'Caption blocked: '+row.query_id)
                require(row.modification_text==caption_text(row.caption_1,row.caption_2) and row.triplet_key_sha256==fingerprint([row.category,row.reference_image_id,row.target_image_id,row.caption_1,row.caption_2]),'Caption policy/triplet changed')
                if warning:warnings.append({'query_id':row.query_id,'reason':warning})
            if name.startswith('official_'):
                counts=self.protocol['actual_counts']['official_val' if split=='val' else 'official_train']
                require(f.groupby('category').size().to_dict()==counts,'Official annotation counts differ')
                for c in CATEGORIES:require(sorted(f[f.category==c].source_row.astype(int))==list(range(counts[c])),'Official annotation rows missing/duplicated')
            write_json(ROOT/'data_audit'/(name+'_warnings.json'),warnings)
        else:
            f['gallery_index']=f.gallery_index.astype(int)
            for c in CATEGORIES:
                g=f[f.category==c];require(g.image_id.is_unique and g.gallery_index.tolist()==list(range(len(g))),'Gallery order/IDs differ')
                if name=='official_val_gallery':require(fingerprint(g.image_id.tolist())==self.protocol['id_order_sha256']['official_val_gallery'][c],'Official image_splits order differs')
        self.frames[name]=f;self.views[name]=view;self.hashes[name]=file_hash(path);return f
    def data(self,phase):
        require(phase in ('development','final','benchmark'),'Unknown data phase')
        if phase=='development':names=('development_train','development_val','development_gallery')
        elif phase=='final':names=('official_train_queries',None,None)
        else:names=(None,'official_val_queries','official_val_gallery')
        train,val,gallery=[self.read(x) if x else None for x in names]
        for name in names:
            if name:require(frame_sha(self.frames[name])==PREPARATION_ANCHORS[name+'.csv'],'In-memory manifest changed after validation: '+name)
        if val is not None:
            for c in CATEGORIES:require(set(val[val.category==c].target_image_id)<=set(gallery[gallery.category==c].image_id),'Target coverage missing in category gallery')
        return {'phase':phase,'train':train,'validation':val,'gallery':gallery,'train_sha256':frame_sha(train) if train is not None else None,'validation_sha256':frame_sha(val) if val is not None else None,'gallery_sha256':frame_sha(gallery) if gallery is not None else None,'package':self.binding,'views':{role:self.views[name] for role,name in zip(('train','validation','gallery'),names) if name}}
    def image_source(self):
        root=self.root/'features'/self.meta['fingerprint']
        return {'prep':self.root,'meta':self.meta,'mapping_hashes':{k:file_hash(root/self.meta['mapping_files'][k]) for k in ('reference','target')}}

class Tokens:
    def __init__(self,frame,tokenizer):
        require(frame.query_id.is_unique,'Duplicate token-cache query IDs');self.rows={};self.tokenizer=tokenizer
        for start in range(0,len(frame),256):
            rows=frame.iloc[start:start+256];sentences=rows.modification_text.tolist()
            lengths=[len(v) for v in tokenizer(sentences,padding=False,truncation=False)['input_ids']];require(max(lengths)<=77,'Overlength caption; do not truncate/drop')
            values=tokenizer(sentences,padding='max_length',max_length=77,truncation=False,return_tensors='pt')
            eos_positions(values['input_ids'],values['attention_mask'],tokenizer.eos_token_id)
            for i,q in enumerate(rows.query_id):self.rows[q]={k:values[k][i].clone() for k in ('input_ids','attention_mask')}
    def take(self,ids):
        require(all(q in self.rows for q in ids),'Token ID coverage missing')
        return {k:torch.stack([self.rows[q][k] for q in ids]).to(DEVICE) for k in ('input_ids','attention_mask')}

class Images:
    def __init__(self,package,base):
        self.base=base;source=package.image_source();self.ref=PreparationStore(source,'reference');self.target=PreparationStore(source,'target');self.package=package
        self.vision_hash=immutable_backbone(base);self.pool=None
    def prepare(self,reference_ids,target_ids,folder):
        refs=sorted(set(reference_ids));targets=sorted(set(target_ids));started=time.perf_counter()
        require(set(refs)<=set(self.ref.mapping) and set(targets)<=set(self.target.mapping),'Image feature coverage missing; no zeros/drop')
        # Full local content validation, once per used shard before any batch hot path.
        for reader,ids in ((self.ref,refs),(self.target,targets)):
            for start in range(0,len(ids),128):
                a=reader.take(ids[start:start+128]);require(np.isfinite(a).all() and np.all(np.linalg.norm(a.reshape(len(a),-1),axis=-1)>1e-8),'Image features zero/nonfinite')
        contract={'schema':'final-reference-CLS-postLN-project-unit-v1','reference_ids_sha256':fingerprint(refs),'cache_shard_size':CFG['cache_shard_size'],'image_contract':self.package.contract,'source_index':self.package.meta['store_index_sha256']['reference'],'vision_hashes':self.vision_hash,'runtime':physical_runtime()}
        specs={'reference_pooled':{'ids':refs,'shape':[512],'dtype':'float32'}}
        self.pool=TrialStore(ROOT/'shared_reference_pool',LOCAL/'shared_reference_pool',contract,specs,CFG['cache_shard_size'])
        def compute(ids):
            values=[]
            for start in range(0,len(ids),CFG['encode_batch_size']):
                a=torch.from_numpy(self.ref.take(ids[start:start+CFG['encode_batch_size']])).to(DEVICE)
                with torch.inference_mode():v=unit(self.base.visual_projection(self.base.vision_model.post_layernorm(a[:,0])))
                values.append(v.cpu().numpy())
            return np.concatenate(values)
        self.pool.prepare('reference_pooled',compute)
        self.refs=set(refs);self.targets=set(targets)
        write_json(Path(folder)/'image_feature_audit.json',{'passed':True,'reference_ids':len(refs),'target_gallery_ids':len(targets),'reference_pool':self.pool.key,'preparation_feature_fingerprint':self.package.meta['fingerprint'],'all_used_local_shards_verified':True,'elapsed_seconds':time.perf_counter()-started,'no_multilayer_vision_cache':True})
    def take(self,role,ids):
        if role=='reference':require(set(ids)<=self.refs,'Unprepared reference IDs');return self.ref.take(ids)
        if role=='reference_pooled':return self.pool.take(role,ids)
        require(role=='target_pooled' and set(ids)<=self.targets,'Unprepared target/gallery IDs');return self.target.take(ids)

class TextFeatures:
    def __init__(self,folder,data,encoder,asset,tokens):
        require(asset['kind']=='encoder' and asset['phase']==('development' if data['phase']=='development' else 'final'),'Development/final text cache encoder phase mismatch')
        require(all(not p.requires_grad for p in encoder.parameters()) and not encoder.training,'Freeze trained encoder before text extraction')
        frame=pd.concat([x for x in (data['train'],data['validation']) if x is not None],ignore_index=True).sort_values('query_id')
        ids=frame.query_id.tolist();require(len(set(ids))==len(ids),'Text train/validation overlap')
        contract={'schema':'trained-conventional-finalLN-padded-zero-v1','phase':data['phase'],'encoder_checkpoint_sha256':asset['sha256'],'encoder_weights_sha256':tensor_hash(encoder),'train_sha256':data['train_sha256'],'validation_sha256':data['validation_sha256'],'caption_policy':CAPTION_POLICY,'runtime':physical_runtime(),'encode_batch_size':CFG['encode_batch_size'],'cache_shard_size':CFG['cache_shard_size'],'precision':'float32','max_length':77}
        specs={'text':{'ids':ids,'shape':[77,512],'dtype':'float32'},'mask':{'ids':ids,'shape':[77],'dtype':'int64'},'eos':{'ids':ids,'shape':[],'dtype':'int64'},'input_ids':{'ids':ids,'shape':[77],'dtype':'int64'}}
        if CFG.get('enable_training') or CFG.get('enable_benchmark'):
            payload=len(ids)*(77*512*4+77*8*2+8)
            key=fingerprint(contract);existing=sum(p.stat().st_size for p in (ROOT/'text_cache'/key).rglob('*.npy')) if (ROOT/'text_cache'/key).exists() else 0
            resource_preflight(folder,'adapter',len(ids),missing_cache_bytes=max(0,payload-existing))
        self.store=TrialStore(ROOT/'text_cache',LOCAL/'text_cache',contract,specs,CFG['cache_shard_size']);self.key=self.store.key
        def compute(qids,role):
            values=[]
            for i in range(0,len(qids),CFG['encode_batch_size']):
                mod=tokens.take(qids[i:i+CFG['encode_batch_size']])
                with torch.inference_mode():
                    if role=='text':value=encoder.encode_text(mod)[0]
                    elif role=='eos':value=eos_positions(mod['input_ids'],mod['attention_mask'],tokens.tokenizer.eos_token_id)
                    else:value=mod['input_ids' if role=='input_ids' else 'attention_mask']
                values.append(value.cpu().numpy())
            return np.concatenate(values)
        for role in specs:self.store.prepare(role,lambda q,r=role:compute(q,r))
        for start in range(0,len(ids),128):
            q=ids[start:start+128];m=torch.from_numpy(self.take('mask',q));i=torch.from_numpy(self.take('input_ids',q));e=torch.from_numpy(self.take('eos',q));t=self.take('text',q)
            require(torch.equal(eos_positions(i,m,tokens.tokenizer.eos_token_id),e),'Text cache EOS/ID alignment')
            require(np.count_nonzero(t[m.numpy()==0])==0 and np.all(np.linalg.norm(t[np.arange(len(q)),e.numpy()],axis=-1)>1e-8),'Text padding/EOS vector invalid')
        write_json(Path(folder)/'text_cache_audit.json',{'passed':True,'fingerprint':self.key,'phase':data['phase'],'queries':len(ids),'encoder_checkpoint':asset,'all_shards_checksum_shape_finite_mask_EOS_verified':True})
    def take(self,role,ids):return self.store.take(role,ids)

def prepare_inputs(data,images,folder):
    frames=[x for x in (data['train'],data['validation']) if x is not None];queries=pd.concat(frames,ignore_index=True)
    targets=set(data['train'].target_image_id) if data['train'] is not None else set()
    if data['gallery'] is not None:targets|=set(data['gallery'].image_id)
    for role,view in data.get('views',{}).items():
        if 'query_id' in view:
            for row in view.itertuples():
                require(int(row.reference_feature_index)==images.ref.mapping[row.reference_image_id] and int(row.target_feature_index)==images.target.mapping[row.target_image_id],'Feature view index/ID mapping differs')
        else:
            for row in view.itertuples():require(int(row.target_feature_index)==images.target.mapping[row.image_id],'Gallery feature index/ID mapping differs')
    images.prepare(queries.reference_image_id,targets,folder)

def batch_for(rows,kind,images,tokens,text=None,training=True):
    ids=rows.query_id.tolist();refs=rows.reference_image_id.tolist()
    reference={'pooled':torch.from_numpy(images.take('reference_pooled',refs)).to(DEVICE)}
    b={'query_ids':ids,'reference_features':reference}
    if kind=='encoder':b['modifier']=tokens.take(ids)
    else:
        reference['tokens']=torch.from_numpy(images.take('reference',refs)).to(DEVICE)
        b.update(text_features=torch.from_numpy(text.take('text',ids)).to(DEVICE),mask=torch.from_numpy(text.take('mask',ids)).to(DEVICE),eos=torch.from_numpy(text.take('eos',ids)).to(DEVICE))
    if training:b.update(target=torch.from_numpy(images.take('target_pooled',rows.target_image_id.tolist())).to(DEVICE),target_ids=rows.target_image_id.tolist())
    return b

def query_forward(model,batch,kind):
    if kind=='encoder':return model.encode_query(batch['reference_features'],batch['modifier'])
    return model.encode_query(batch['reference_features'],batch['text_features'],batch['mask'],batch['eos'])


In [ ]:
# evaluate.py — embedded runtime
@torch.inference_mode()
def retrieval_evaluate(model,data,images,batch_fn,kind,top_k=50):
    require(data['validation'] is not None and data['gallery'] is not None,'Evaluation manifest required')
    model.eval();per={};queries=[];top=[]
    for c in CATEGORIES:
        f=data['validation'][data['validation'].category==c];g=data['gallery'][data['gallery'].category==c].sort_values('gallery_index',kind='stable')
        require(len(f)>0 and g.gallery_index.tolist()==list(range(len(g))) and g.image_id.is_unique,'Gallery/category alignment invalid')
        gids=g.image_id.tolist();lookup={v:i for i,v in enumerate(gids)};vectors=images.take('target_pooled',gids);vectors=vectors/np.linalg.norm(vectors,axis=-1,keepdims=True)
        ranks=[]
        for start in range(0,len(f),CFG['eval_batch_size']):
            rows=f.iloc[start:start+CFG['eval_batch_size']];b=batch_fn(rows,False);require('target' not in b,'Target entered query inference')
            q=query_forward(model,b,kind).cpu().numpy();q=q/np.linalg.norm(q,axis=-1,keepdims=True)
            positions=np.array([lookup[x] for x in rows.target_image_id]);scores=q@vectors.T;require(np.isfinite(scores).all(),'Nonfinite scores')
            ts=scores[np.arange(len(rows)),positions];r=1+(scores>ts[:,None]).sum(1)+((scores==ts[:,None])&(np.arange(len(gids))[None,:]<positions[:,None])).sum(1)
            ranks.extend(r.tolist());orders=np.argsort(-scores,axis=1,kind='stable')[:,:min(top_k,len(gids))]
            for i,row in enumerate(rows.itertuples()):
                queries.append({'query_id':row.query_id,'category':c,'reference_image_id':row.reference_image_id,'target_image_id':row.target_image_id,'target_rank':int(r[i])})
                top.extend({'query_id':row.query_id,'category':c,'rank':j+1,'image_id':gids[pos],'score':float(scores[i,pos])} for j,pos in enumerate(orders[i]))
        per[c]={f'R@{k}':float((np.asarray(ranks)<=k).mean()*100) for k in (1,5,10,50)}
    require(len(queries)==len(data['validation']) and len({q['query_id'] for q in queries})==len(queries),'Evaluation query coverage differs')
    return {'scope':data['phase'],'n_query':len(queries),'query_counts':data['validation'].groupby('category').size().to_dict(),'gallery_counts':data['gallery'].groupby('category').size().to_dict(),'unit':'percent','per_category':per,**macro_metrics(per)},pd.DataFrame(queries),pd.DataFrame(top)

def metrics_from_predictions(queries,data):
    f=data['validation'];g=data['gallery'];keys=['query_id','category','reference_image_id','target_image_id']
    require(len(queries)==len(f) and queries.query_id.is_unique and set(queries.query_id)==set(f.query_id),'Prediction query coverage')
    require(len(queries.merge(f[keys],on=keys,validate='one_to_one'))==len(f),'Prediction source ID mapping differs')
    per={}
    for c in CATEGORIES:
        r=queries[queries.category==c].target_rank.to_numpy(dtype=np.float64);ng=len(g[g.category==c])
        require(len(r)==int((f.category==c).sum()) and np.isfinite(r).all() and np.all(r==r.astype(int)) and np.all((r>=1)&(r<=ng)),'Saved ranks invalid')
        per[c]={f'R@{k}':float((r<=k).mean()*100) for k in (1,5,10,50)}
    return {'scope':data['phase'],'n_query':len(f),'query_counts':f.groupby('category').size().to_dict(),'gallery_counts':g.groupby('category').size().to_dict(),'unit':'percent','per_category':per,**macro_metrics(per)}

def metric_equal(a,b):
    require(a['n_query']==b['n_query'] and all(abs(a['per_category'][c][f'R@{k}']-b['per_category'][c][f'R@{k}'])<1e-10 for c in CATEGORIES for k in (1,5,10,50)) and abs(a['Mean']-b['Mean'])<1e-10,'Saved metric mismatch')
    for k in (1,5,10,50):require(abs(a['overall'][f'R@{k}']-b['overall'][f'R@{k}'])<1e-10,'Saved macro mismatch')

def verify_topk(top,queries,data,top_k):
    require(set(top.query_id)==set(queries.query_id) and not top.duplicated(['query_id','rank']).any() and not top.duplicated(['query_id','image_id']).any(),'TopK ID/duplicate coverage')
    require(len(top.merge(queries[['query_id','category']],on=['query_id','category'],validate='many_to_one'))==len(top),'TopK category mapping')
    require(np.isfinite(top.score).all(),'TopK NaN/Inf')
    for c in CATEGORIES:
        g=data['gallery'][data['gallery'].category==c];p=top[top.category==c];n=int((queries.category==c).sum());k=min(top_k,len(g))
        require(len(p)==n*k and set(p.image_id)<=set(g.image_id),'TopK gallery/count differs')
        require(p.groupby('query_id')['rank'].apply(lambda r:sorted(r)==list(range(1,k+1))).all(),'TopK rank coverage differs')
        ordered=p.sort_values(['query_id','rank'],kind='stable');same=ordered.query_id.eq(ordered.query_id.shift());previous=ordered.score.shift()
        require((ordered.loc[same,'score']<=previous[same]).all(),'TopK scores are not descending')
        gallery_order=dict(zip(g.image_id,g.gallery_index));indices=ordered.image_id.map(gallery_order);ties=same & ordered.score.eq(previous)
        require((indices[ties]>indices.shift()[ties]).all(),'TopK exact ties differ from locked gallery order')
        matched=p.merge(queries[['query_id','target_image_id','target_rank']],on='query_id',validate='many_to_one');hits=matched[matched.image_id==matched.target_image_id]
        expected=queries[(queries.category==c)&(queries.target_rank<=k)]
        require(set(hits.query_id)==set(expected.query_id) and (hits['rank']==hits.target_rank).all(),'TopK target positions disagree with saved ranks')

def export_evaluation(folder,model,data,images,batch_fn,kind):
    folder=Path(folder);metric,q,top=retrieval_evaluate(model,data,images,batch_fn,kind,CFG['export_top_k'])
    write_csv(folder/'per_query_results.csv',q);write_csv(folder/'top_k_results.csv',top)
    per=[{'category':c,**m,'Mean':(m['R@10']+m['R@50'])/2} for c,m in metric['per_category'].items()]+[{'category':'Overall macro',**metric['overall'],'Mean':metric['Mean']}]
    write_csv(folder/'per_category_metrics.csv',pd.DataFrame(per));write_json(folder/'metrics.json',metric)
    saved=pd.read_csv(folder/'per_query_results.csv',dtype={'query_id':str});savedtop=pd.read_csv(folder/'top_k_results.csv',dtype={'query_id':str,'image_id':str})
    metric_equal(metrics_from_predictions(saved,data),metric);verify_topk(savedtop,saved,data,CFG['export_top_k'])
    return metric,{n:file_hash(folder/n) for n in ('per_query_results.csv','top_k_results.csv','per_category_metrics.csv','metrics.json')}


In [ ]:
# engine.py — embedded runtime
def required_checkpoint(state,binding):
    keys={'signature','binding','phase','seed','kind','model','optimizer','scheduler','scaler','rng','epoch','cursor','step','history','metrics','finished','best_epoch','best_mean','bad_epochs','timing','initial_sha256'}
    require(keys<=state.keys(),'Incomplete checkpoint: '+','.join(sorted(keys-state.keys())))
    require(state['binding']==binding and state['signature']==fingerprint(binding),'Resume config/code/data/runtime mismatch')
    require(state['scheduler'] is None and state['scaler'] is None and {'state','param_groups'}<=state['optimizer'].keys() and {'python','numpy','torch','cuda'}<=state['rng'].keys(),'Missing optimizer/RNG or incompatible scheduler/scaler')
    require(len(state['history'])==state['step'] and state['phase']==binding['phase'] and state['seed']==binding['seed'] and state['kind']==binding['kind'],'Checkpoint progress/phase mismatch')
    require(state['epoch']>=0 and state['cursor']>=0 and state['step']>=0,'Negative checkpoint cursor')

def optimize_stage(model,frame,batch_fn,eval_fn,folder,local,binding,hp,budget,patience=None):
    """Shared production runner. Tests call this core with temporary models/frames/folders only."""
    folder,local=Path(folder),Path(local);folder.mkdir(parents=True,exist_ok=True);local.mkdir(parents=True,exist_ok=True)
    signature=fingerprint(binding);write_json(folder/'training_binding.json',{'signature':signature,'binding':binding},immutable=True)
    initial=tensor_hash(model);params=[p for p in model.parameters() if p.requires_grad];require(params,'No trainable parameters')
    opt=torch.optim.AdamW(params,lr=hp['learning_rate'],weight_decay=hp['weight_decay'],betas=(.9,.999),eps=1e-8)
    state={'signature':signature,'binding':binding,'phase':binding['phase'],'seed':binding['seed'],'kind':binding['kind'],'epoch':0,'cursor':0,'step':0,'history':[],'metrics':[],'finished':False,'stop_reason':None,'best_epoch':None,'best_mean':None,'bad_epochs':0,'initial_sha256':initial,'scheduler':None,'scaler':None,'timing':{'compute_seconds':0.,'training_wall_seconds':0.,'validation_seconds':0.,'checkpoint_seconds':0.},'best_snapshot':None}
    candidates=[]
    for path in (local/'last.pt',folder/'last.pt'):
        if path.exists():
            candidate=checkpoint_load(path,signature);required_checkpoint(candidate,binding);require(candidate['initial_sha256']==initial,'Resume initializer mismatch');candidates.append(candidate)
    if candidates:
        state=max(candidates,key=lambda x:(x['step'],x['epoch'],len(x['metrics']),x['finished']));model.load_state_dict(state['model'],strict=True);opt.load_state_dict(state['optimizer']);restore_rng(state['rng'])
    else:seed_all(binding['seed'])
    del candidates
    def persist(improved=False):
        started=time.perf_counter();state.update(model=cpu_tree(model.state_dict()),optimizer=cpu_tree(opt.state_dict()),rng=rng_state())
        if improved:state['best_snapshot']=cpu_tree({k:v for k,v in state.items() if k!='best_snapshot'})
        checkpoint_save(local/'last.pt',state,signature);sync_checkpoint(local/'last.pt',folder/'last.pt',signature)
        if state.get('best_snapshot') is not None:
            best=state['best_snapshot'];expected=read_json(folder/'best.json') if (folder/'best.json').exists() else None
            if expected is None or expected.get('step')!=best['step'] or not (folder/'best.pt').is_file():
                checkpoint_save(local/'best.pt',best,signature);sync_checkpoint(local/'best.pt',folder/'best.pt',signature)
        if state['finished'] and binding['phase']=='final':sync_checkpoint(local/'last.pt',folder/'final.pt',signature)
        state['timing']['checkpoint_seconds']+=time.perf_counter()-started
        write_json(folder/'status.json',{k:state[k] for k in ('phase','kind','seed','epoch','cursor','step','finished','stop_reason','best_epoch','best_mean','timing','initial_sha256')})
        write_csv(folder/'training_history.csv',pd.DataFrame(state['history']));write_json(folder/'metric_history.json',state['metrics'])
    indexed=frame.set_index('query_id',drop=False)
    try:
        for epoch in range(state['epoch'],budget):
            if state['finished']:break
            order=epoch_batches(frame,hp['batch_size'],binding['seed'],epoch);require(state['cursor']<=len(order),'Resume cursor beyond epoch')
            for cursor in range(state['cursor'],len(order)):
                wall=time.perf_counter();ids=order[cursor];rows=indexed.loc[ids];require(rows.query_id.tolist()==ids and set(rows.official_split)=={'train'},'Training query order/split differs')
                batch=batch_fn(rows,True);model.train();opt.zero_grad(set_to_none=True)
                if next(model.parameters()).is_cuda:torch.cuda.synchronize()
                t=time.perf_counter();query=query_forward(model,batch,binding['kind']);loss=retrieval_loss(query,batch['target'],batch['target_ids'],hp['retrieval_temperature']);finite(loss,'training loss');loss.backward()
                require(all(p.grad is not None for p in params),'Trainable branch missing gradient')
                grad=torch.nn.utils.clip_grad_norm_(params,hp['gradient_clip'],error_if_nonfinite=True);opt.step()
                require(bool(torch.stack([torch.isfinite(p).all() for p in params]).all()),'Optimizer produced nonfinite weight')
                if next(model.parameters()).is_cuda:torch.cuda.synchronize()
                compute=time.perf_counter()-t;elapsed=time.perf_counter()-wall
                values=torch.stack([loss.detach(),grad.detach()]).cpu().tolist();state['step']+=1;state['cursor']=cursor+1
                state['timing']['compute_seconds']+=compute;state['timing']['training_wall_seconds']+=elapsed
                state['history'].append({'epoch':epoch+1,'step':state['step'],'cursor':cursor+1,'batch_ids_sha256':fingerprint(ids),'loss_retrieval':values[0],'gradient_norm':values[1],'compute_seconds':compute,'training_wall_seconds':elapsed})
                if state['step']%hp['checkpoint_every_steps']==0:persist()
            improved=False
            if eval_fn is not None:
                model.eval();t=time.perf_counter();metric,pred=eval_fn(model);seconds=time.perf_counter()-t;state['timing']['validation_seconds']+=seconds
                require(metric['scope']=='development','Only internal development validation may choose epochs')
                improved=state['best_mean'] is None or metric['Mean']>state['best_mean']
                if improved:state.update(best_epoch=epoch+1,best_mean=metric['Mean'],bad_epochs=0)
                else:state['bad_epochs']+=1
                write_csv(folder/f'epoch_{epoch+1:02d}_per_query.csv',pred)
                state['metrics'].append({'epoch':epoch+1,'metrics':metric,'validation_seconds':seconds,'predictions_sha256':file_hash(folder/f'epoch_{epoch+1:02d}_per_query.csv')})
            require(binding['phase']=='development' or (eval_fn is None and patience is None),'Final training must not evaluate/early-stop on official-val')
            done=(epoch+1>=budget or (patience is not None and state['bad_epochs']>=patience))
            state.update(epoch=epoch+1,cursor=0,finished=done,stop_reason=('patience' if patience is not None and state['bad_epochs']>=patience else 'fixed_budget') if done else None)
            persist(improved);print(binding['phase'],binding['kind'],'seed',binding['seed'],'epoch',epoch+1,'step',state['step'],'Mean',state['best_mean'])
        # A lost best upload can be reconstructed from the authoritative last snapshot without optimizer updates.
        if state['finished'] and not (folder/('final.pt' if binding['phase']=='final' else 'best.pt')).exists():persist()
        return {k:state[k] for k in ('finished','epoch','step','best_epoch','best_mean','metrics','timing','initial_sha256','stop_reason')}
    except BaseException:
        write_bytes(folder/'training_traceback.txt',traceback.format_exc().encode());raise
    finally:opt.zero_grad(set_to_none=True)


In [ ]:
# provenance.py — embedded runtime
def data_identity(data):return {k:data[k] for k in ('phase','train_sha256','validation_sha256','gallery_sha256','package')}
def semantic_development(package,seed):return {'schema':SCHEMA,'seed_scope':'full_pipeline','seed':seed,'encoder_seed':seed,'adapter_seed':seed,'encoder_recipe':encoder_recipe(),'adapter_recipe':adapter_recipe(),'science_sha256':SCIENCE_SHA256,'data':data_identity(package.data('development'))}

def asset_record(path,phase,kind,seed,epochs,fmt='native',encoder_sha=None):
    path=Path(path);marker=read_json(path.with_suffix('.json'));require(file_hash(path)==marker['sha256'],'Asset checksum mismatch: '+str(path))
    return {'path':str(path),'sha256':marker['sha256'],'signature':marker['signature'],'format':fmt,'phase':phase,'kind':kind,'seed':seed,'epochs':epochs,'encoder_checkpoint_sha256':encoder_sha}

def validate_asset(record):
    require(record['format'] in ('native','legacy_conventional_v1','legacy_adapter_v2'),'Unknown checkpoint schema')
    path=Path(record['path']);m=read_json(path.with_suffix('.json'));require(m['sha256']==record['sha256'] and m['signature']==record['signature'] and file_hash(path)==record['sha256'],'Checkpoint marker/content changed: '+str(path))
    return committed_checkpoint_load(path,record['signature'])

def load_encoder_asset(record,base,tokenizer):
    require(record['kind']=='encoder','An adapter checkpoint cannot be used as encoder')
    state=validate_asset(record)
    if record['format']=='legacy_conventional_v1':
        require(record['seed']==42 and record['phase']=='development' and state['phase']=='conventional' and state['config']['seed']==42 and state['epoch']==3 and state['finished'],'Legacy encoder phase/config differs')
        roots={k.split('.')[0] for k in state['model']};require(roots=={'text','projection','align','inject','merge','align_norm','align_projection'},'Unknown legacy encoder module roots')
        weights={k:v for k,v in state['model'].items() if k.startswith(('text.','projection.'))}
    else:
        required_checkpoint(state,state['binding']);require(state['kind']=='encoder' and state['phase']==record['phase'] and state['seed']==record['seed'] and state['binding']['origin']=='locked_pretrained_fresh','Encoder phase/seed/origin mismatch')
        require(state['binding']['initial_pretrained_sha256']==EXTRACTION_CONTRACT['encoder_weight_sha256'],'Final/development encoder did not start from locked pretrained')
        weights=state['model']
    model=fresh_encoder(base,tokenizer,record['seed']);model.load_state_dict(weights,strict=True);model.freeze()
    if record['format']=='legacy_conventional_v1':require(tensor_hash(model)==LEGACY['encoder_text_hash'],'Extracted legacy conventional weights differ')
    del state;return model

def load_adapter_asset(record,encoder):
    require(record['kind']=='adapter' and all(not p.requires_grad for p in encoder.parameters()),'Adapter expects frozen trained encoder')
    state=validate_asset(record)
    if record['format']=='legacy_adapter_v2':
        require(state['binding']['base_checkpoint']==LEGACY['encoder_best']['sha256'] and state['binding']['base_seed']==state['binding']['adapter_seed']==record['seed']==42,'Legacy adapter encoder/seed mismatch')
    else:
        required_checkpoint(state,state['binding']);require(state['phase']==record['phase'] and state['kind']=='adapter' and state['seed']==record['seed'] and state['binding']['encoder_asset']['sha256']==record['encoder_checkpoint_sha256'],'Adapter final/development provenance mismatch')
    model=fresh_adapter(encoder,record['seed']);model.load_state_dict(state['model'],strict=True)
    require(torch.equal(model.projection_weight,encoder.projection.weight.detach()),'Adapter frozen projection belongs to another encoder')
    model.eval();del state;return model

def verify_epoch_history(folder,history,data):
    require(history and [x['epoch'] for x in history]==list(range(1,len(history)+1)),'Epoch metrics missing/non-contiguous; never interpolate')
    receipts={};norm=[]
    for item in history:
        require(item['metrics'].get('scope','') in ('development','fixed full development validation only','full locked development validation; NOT official-val'),'History is not development validation')
        path=Path(folder)/f'epoch_{item["epoch"]:02d}_per_query.csv';q=pd.read_csv(path,dtype={'query_id':str});metric=metrics_from_predictions(q,data);metric_equal(metric,item['metrics'])
        sha=file_hash(path)
        if 'predictions_sha256' in item:require(sha==item['predictions_sha256'],'Epoch prediction SHA differs')
        receipts[str(path)]=sha;norm.append({'epoch':item['epoch'],'metrics':metric,'predictions_sha256':sha})
    return norm,receipts

def verify_receipts(receipts):
    for name,sha in receipts.items():require(file_hash(Path(name))==sha,'Artifact changed or missing: '+name)

def check_development_payload(state,asset,record,kind):
    seed=record['seed']
    if asset['format']=='legacy_conventional_v1':
        require(kind=='encoder' and seed==42 and state['phase']=='conventional' and state['config']['seed']==42,'Legacy encoder cannot be relabeled as another pipeline seed')
    elif asset['format']=='legacy_adapter_v2':
        require(kind=='adapter' and seed==42 and state['binding']['base_seed']==state['binding']['adapter_seed']==42 and state['binding']['base_checkpoint']==record['encoder']['sha256'],'Legacy adapter seed/encoder mismatch')
    else:
        required_checkpoint(state,state['binding']);b=state['binding']
        require(state['phase']=='development' and state['kind']==kind and state['seed']==b['encoder_seed']==b['adapter_seed']==seed,'Checkpoint payload is not the claimed full-pipeline seed')
        require(b['data']==record['semantic']['data'] and b['science_sha256']==SCIENCE_SHA256 and b['recipe']==(encoder_recipe() if kind=='encoder' else adapter_recipe()),'Checkpoint development semantics differ')
        if kind=='adapter':require(b['encoder_asset']['sha256']==record['encoder']['sha256'],'Adapter belongs to another development encoder')

def verify_development_record(record,package,check_encoder=True):
    seed=record['seed'];require(record['phase']=='development' and record['semantic']==semantic_development(package,seed),'Development semantic/data/seed recipe differs')
    require(record['encoder']['seed']==record['adapter']['seed']==seed and record['encoder']['phase']==record['adapter']['phase']=='development','Not a full-pipeline seed result')
    verify_receipts(record['file_receipts']);data=package.data('development')
    for key in ('encoder_history','adapter_history'):
        require(record[key] and all(x['metrics']['scope']=='development' and x['metrics']['n_query']==1796 for x in record[key]),'Full internal-val history required')
    for kind in ('encoder','adapter'):
        a=record[kind];require(a['kind']==kind and file_hash(a['path'])==a['sha256'],'Development checkpoint hash differs')
        m=read_json(Path(a['path']).with_suffix('.json'));require(m['signature']==a['signature'] and m['sha256']==a['sha256'],'Development marker differs')
    best=record['adapter_best_epoch'];means=[x['metrics']['Mean'] for x in record['adapter_history']];expected=next(i+1 for i,v in enumerate(means) if v==max(means));require(best==expected,'Best/earliest-tie selection differs')
    q=pd.read_csv(record['per_query_path'],dtype={'query_id':str});metric_equal(metrics_from_predictions(q,data),record['adapter_metrics'])
    top=pd.read_csv(record['topk_path'],dtype={'query_id':str,'image_id':str});verify_topk(top,q,data,CFG['export_top_k'])
    if check_encoder:
        state=validate_asset(record['encoder']);check_development_payload(state,record['encoder'],record,'encoder');require(state['epoch']==record['encoder']['epochs'],'Selected encoder epoch differs');del state
    for kind in ('encoder','adapter'):
        last=validate_asset(record[kind+'_last']);check_development_payload(last,record[kind+'_last'],record,kind);require(last['finished'] and last['epoch']==len(record[kind+'_history']) and last['cursor']==0,'Incomplete last checkpoint or history')
        expected_steps=sum(len(epoch_batches(data['train'],CFG[kind]['batch_size'],seed,e)) for e in range(last['epoch']))
        require(last['step']==expected_steps and len(last['history'])==expected_steps and {'optimizer','rng'}<=last.keys(),'Last optimizer/RNG/full-train coverage missing')
        del last
    state=validate_asset(record['adapter']);check_development_payload(state,record['adapter'],record,'adapter');require(state['best_epoch']==best and state['epoch']==best,'Wrong adapter best checkpoint')
    require(sum(v.numel() for k,v in state['model'].items() if k!='projection_weight')==446978,'Adapter state parameter count differs')
    if record['adapter']['format']=='native':required_checkpoint(state,state['binding'])
    del state;return record

def import_seed42(package):
    folder=ROOT/'development'/'seed_42';folder.mkdir(parents=True,exist_ok=True);out=folder/'result.json'
    if out.exists():return verify_development_record(read_json(out),package)
    conv=Path(CFG['legacy_conventional_root'])/'conventional';adapter=Path(CFG['legacy_adapter_root']);data=package.data('development')
    require(file_hash(conv/'config.json')==LEGACY['encoder_config_sha256'] and read_json(conv/'best.json')==LEGACY['encoder_best'],'Known legacy encoder config/marker changed')
    cfg=read_json(conv/'config.json');e=cfg['config'];require(cfg['variant']=='conventional' and e['seed']==42 and all(e[k]==CFG['encoder'][k] for k in ('batch_size','learning_rate','weight_decay','gradient_clip','retrieval_temperature')) and e['max_epochs']==3 and e['amp'] is False,'Legacy encoder science differs')
    require(file_hash(adapter/'training_config.json')==LEGACY['adapter_config_sha256'],'Known legacy adapter training contract changed')
    training=read_json(adapter/'training_config.json');binding=training['binding'];a=binding['config']
    require(training['signature']==fingerprint(binding) and binding['code_sha256']==LEGACY['adapter_code_sha256'] and binding['objective']=='symmetric multi-positive InfoNCE only','Legacy adapter schema/code/objective unrecognized')
    require(binding['base_seed']==binding['adapter_seed']==42 and binding['base_checkpoint']==LEGACY['encoder_best']['sha256'] and binding['initial_sha256']==LEGACY['adapter_initial_sha256'],'Legacy adapter origin differs')
    for old,new in (('batch_size','batch_size'),('learning_rate','learning_rate'),('weight_decay','weight_decay'),('gradient_clip','gradient_clip'),('retrieval_temperature','retrieval_temperature'),('max_epochs','max_epochs'),('patience','patience'),('adapter_width','width'),('adapter_heads','heads')):require(a[old]==CFG['adapter'][new],'Legacy adapter HP differs: '+old)
    require(a['amp'] is False and a['tf32'] is False and training['scheduler'] is None and training['scaler'] is None and training['trainable_parameters']==446978,'Legacy precision/optimizer differs')
    for key in ('train_sha256','validation_sha256','gallery_sha256'):require(binding[key]==data[key],'Legacy manifest differs: '+key)
    require(read_json(adapter/'best.json')==LEGACY['adapter_best'] and read_json(adapter/'last.json')==LEGACY['adapter_last'],'Legacy checkpoint markers differ')
    status=read_json(adapter/'status.json');result=read_json(adapter/'best_result_verification.json');require(status['finished'] and result['passed'] and result['query_coverage']==1796 and result['checkpoint_sha256']==LEGACY['adapter_best']['sha256'],'Legacy completion/result incomplete')
    receipts={}
    for name,sha in result['outputs'].items():p=safe_child(adapter,name);require(file_hash(p)==sha,'Legacy output corrupt');receipts[str(p)]=sha
    enc_hist,er=verify_epoch_history(conv,read_json(conv/'metric_history.json'),data);ad_hist,ar=verify_epoch_history(adapter,read_json(adapter/'metric_history.json'),data);receipts.update(er);receipts.update(ar)
    require(len(enc_hist)==3 and len(ad_hist)<=10 and status['epoch']==len(ad_hist),'Legacy epoch coverage differs')
    for p in (conv/'config.json',conv/'metric_history.json',adapter/'training_config.json',adapter/'metric_history.json'):receipts[str(p)]=file_hash(p)
    encoder=asset_record(conv/'best.pt','development','encoder',42,3,'legacy_conventional_v1')
    ad=asset_record(adapter/'best.pt','development','adapter',42,result['best_epoch'],'legacy_adapter_v2',encoder['sha256'])
    # Last must also be intact and completed; it is never silently reinitialized or migrated for resume.
    last=committed_checkpoint_load(adapter/'last.pt',training['signature']);require(last['finished'] and last['epoch']==status['epoch'] and last['step']==status['step'] and len(last['history'])==last['step'] and {'optimizer','rng','cursor'}<=last.keys(),'Legacy last checkpoint incomplete');del last
    receipts[str(adapter/'last.pt')]=LEGACY['adapter_last']['sha256']
    record={'schema':SCHEMA,'phase':'development','seed':42,'semantic':semantic_development(package,42),'encoder':encoder,'adapter':ad,'encoder_last':asset_record(conv/'last.pt','development','encoder',42,3,'legacy_conventional_v1'),'adapter_last':asset_record(adapter/'last.pt','development','adapter',42,status['epoch'],'legacy_adapter_v2',encoder['sha256']),'encoder_history':enc_hist,'adapter_history':ad_hist,'encoder_metrics':enc_hist[2]['metrics'],'adapter_metrics':result['metrics'],'adapter_best_epoch':result['best_epoch'],'file_receipts':receipts,'per_query_path':str(adapter/'per_query_results.csv'),'topk_path':str(adapter/'top_k_results.csv'),'source':'verified read-only semantic import; no training/resume of legacy weights','timing':{'adapter':status,'encoder_observed_epoch_seconds':sum(x.get('epoch_seconds',0) for x in read_json(conv/'metric_history.json')),'encoder_full_wall_seconds':None,'historical_scope':'Encoder epoch counters include train/eval; complete historical pipeline wall time cannot be reconstructed; do not claim pipeline efficiency from adapter compute alone'}}
    verify_development_record(record,package);write_json(out,record,immutable=True);return record


In [ ]:
# phases.py — embedded runtime
def stage_folder(phase,seed,kind):return ROOT/phase/f'seed_{seed}'/kind

def runtime_training_guard():
    # Do not silently upgrade/downgrade torch/CUDA. Match the completed seed42 training environment.
    expected={'torch':'2.14.1+cu130','transformers':'4.57.1','tokenizers':'0.22.1','huggingface-hub':'0.36.0','numpy':'2.4.6'}
    require(all(importlib.metadata.version(k)==v for k,v in expected.items()),'Training runtime differs from verified seed42; use matching Colab environment or an explicitly reviewed new campaign. No torch/CUDA reinstall is attempted')

def resource_preflight(folder,kind,rows,missing_cache_bytes=None):
    require(DEVICE.type=='cuda','Full pipeline training requires CUDA')
    import psutil
    memory=psutil.virtual_memory();require(memory.available>(3 if kind=='encoder' else 1)*1024**3,'Insufficient available RAM before cache/checkpoint allocation')
    disk=shutil.disk_usage(LOCAL);free,total=torch.cuda.mem_get_info();text_bytes=(rows*77*512*4 if missing_cache_bytes is None else missing_cache_bytes) if kind=='adapter' else 0
    local_needed=text_bytes+8*1024**3;drive_needed=text_bytes+(3*1024**3 if kind=='encoder' else 256*1024**2)
    require(disk.free>local_needed,'Insufficient local disk for exact FP32 cache/checkpoints; no automatic batch/precision change')
    require(free>(2 if kind=='adapter' else 3)*1024**3,'Insufficient free VRAM; preserve batch/precision and choose an adequate runtime')
    from google.colab import auth
    from googleapiclient.discovery import build
    import google.auth
    try:credentials,_=google.auth.default();service=build('drive','v3',credentials=credentials,cache_discovery=False);quota=service.about().get(fields='storageQuota').execute()['storageQuota']
    except Exception:
        auth.authenticate_user();credentials,_=google.auth.default();service=build('drive','v3',credentials=credentials,cache_discovery=False);quota=service.about().get(fields='storageQuota').execute()['storageQuota']
    require('limit' in quota and int(quota['limit'])-int(quota.get('usage',0))>drive_needed,'Drive quota insufficient or unavailable for this phase/seed; preserve old artifacts and increase storage before continuing')
    write_json(Path(folder)/'resource_report.json',{'RAM_available_bytes':memory.available,'RAM_total_bytes':memory.total,'disk_free_bytes':disk.free,'CUDA_free_bytes':free,'CUDA_total_bytes':total,'local_reserve_required_bytes':local_needed,'Drive_reserve_required_bytes':drive_needed,'Drive_quota':quota,'precision':'float32','batch_fixed':True,'note':'Per selected phase/seed upper-bound reserve; already valid caches are reused. Encoder cost is retained in pipeline timing.'})

def stage_binding(package,data,phase,seed,kind,cache_key,encoder_asset,lock,budget):
    require(seed in CFG['seeds'],'Unknown seed')
    return {'schema':SCHEMA,'science_sha256':SCIENCE_SHA256,'implementation_sha256':CODE_SHA256,'phase':phase,'kind':kind,'seed':seed,'encoder_seed':seed,'adapter_seed':seed,'seed_scope':'full_pipeline','data':data_identity(data),'recipe':encoder_recipe() if kind=='encoder' else adapter_recipe(),'budget_epochs':budget,'runtime':physical_runtime(),'cache_fingerprint':cache_key,'encoder_asset':encoder_asset,'final_lock_sha256':fingerprint(lock) if lock else None,'origin':'locked_pretrained_fresh' if kind=='encoder' else 'fresh_zero_gate_adapter','initial_pretrained_sha256':package.contract['encoder_weight_sha256']}

def branch_complete(folder,data,phase,seed,kind,encoder_asset=None,lock=None,_candidate=None):
    path=Path(folder)/'result.json'
    if _candidate is None and not path.exists():return None
    record=read_json(path) if _candidate is None else _candidate;require(record['schema']==SCHEMA and record['phase']==phase and record['seed']==seed and record['kind']==kind,'Completed branch identity mismatch')
    require(record['data']==data_identity(data) and record['recipe']==(encoder_recipe() if kind=='encoder' else adapter_recipe()) and record['science_sha256']==SCIENCE_SHA256,'Completed branch scientific semantics differ; preserve and review instead of retraining')
    require(record['encoder_asset']==encoder_asset and record['lock_sha256']==(fingerprint(lock) if lock else None),'Completed parent encoder/final lock differs')
    verify_receipts(record['file_receipts']);state=validate_asset(record['last']);required_checkpoint(state,state['binding']);require(state['seed']==seed and state['kind']==kind and state['phase']==phase and state['binding']['encoder_seed']==state['binding']['adapter_seed']==seed,'Completed checkpoint was relabeled as another phase/seed/kind');require(state['binding']['data']==record['data'] and state['binding']['recipe']==record['recipe'] and state['binding']['science_sha256']==SCIENCE_SHA256 and state['binding']['encoder_asset']==encoder_asset and state['binding']['final_lock_sha256']==record['lock_sha256'] and state['binding']['budget_epochs']==record['budget_epochs'],'Completed checkpoint scientific binding differs');require(state['finished'] and state['epoch']==record['finished_epochs'] and len(state['history'])==state['step'],'Completed branch last checkpoint incomplete')
    expected_steps=sum(len(epoch_batches(data['train'],record['recipe']['batch_size'],seed,e)) for e in range(state['epoch']));require(state['step']==expected_steps and state['cursor']==0,'Training coverage/steps incomplete')
    if phase=='final':require(not state['metrics'] and state['epoch']==record['budget_epochs'],'Final branch evaluated/selects epochs or has wrong fixed budget')
    selected_binding=state['binding'];expected_epoch=state['best_epoch'] if phase=='development' else record['budget_epochs'];del state
    selected=validate_asset(record['asset']);required_checkpoint(selected,selected_binding);require(selected['epoch']==record['asset']['epochs']==expected_epoch,'Selected checkpoint epoch/identity differs')
    if phase=='final':require(record['asset']['sha256']==record['last']['sha256'],'Final checkpoint differs from completed last state')
    del selected
    if phase=='development':
        history,files=verify_epoch_history(folder,record['history'],data);require(history==record['history'],'Development history changed');verify_receipts(files)
        q=pd.read_csv(Path(folder)/'per_query_results.csv',dtype={'query_id':str});metric_equal(metrics_from_predictions(q,data),record['metrics']);verify_topk(pd.read_csv(Path(folder)/'top_k_results.csv',dtype={'query_id':str,'image_id':str}),q,data,CFG['export_top_k'])
    return record

def train_branch(package,phase,seed,kind,base,tokenizer,images,encoder_asset=None,lock=None):
    require(phase in ('development','final') and kind in ('encoder','adapter'),'Unknown train phase/kind')
    data=package.data(phase);require(len(data['train'])==(16204 if phase=='development' else 18000),'Training count differs')
    if phase=='final':require(lock is not None and lock==read_final_lock(package) and data['train_sha256']==lock['final_train_sha256'],'Final lock/data missing or changed')
    folder=stage_folder(phase,seed,kind);folder.mkdir(parents=True,exist_ok=True)
    existing=branch_complete(folder,data,phase,seed,kind,encoder_asset,lock)
    if existing:return existing
    require(CFG['enable_training'] is True,'Training disabled; enable_training=True explicitly')
    runtime_training_guard()
    if kind=='encoder':resource_preflight(folder,kind,len(data['train']))
    previous=folder/'training_binding.json'
    if previous.exists():
        old=read_json(previous)['binding']
        require(old['implementation_sha256']==CODE_SHA256 and old['runtime']==physical_runtime() and old['data']==data_identity(data) and old['encoder_asset']==encoder_asset and old['recipe']==(encoder_recipe() if kind=='encoder' else adapter_recipe()) and old['final_lock_sha256']==(fingerprint(lock) if lock else None),'Active resume incompatible before cache allocation; preserve checkpoint and review')
    with measured(folder,kind+'_input_cache'):
        prepare_inputs(data,images,folder);frame=pd.concat([f for f in (data['train'],data['validation']) if f is not None],ignore_index=True);tokens=Tokens(frame,tokenizer)
        encoder=None;text=None
        if kind=='encoder':
            require(encoder_asset is None,'A new encoder cannot warm-start from a development encoder')
            model=fresh_encoder(base,tokenizer,seed);cache_key=fingerprint({'image_pool':images.pool.key,'tokens_manifest':frame_sha(frame),'tokenizer':package.contract['tokenizer']})
        else:
            require(encoder_asset and encoder_asset['phase']==phase and encoder_asset['seed']==seed,'Adapter must use matching phase/seed encoder')
            encoder=load_encoder_asset(encoder_asset,base,tokenizer);encoder.freeze();text=TextFeatures(folder,data,encoder,encoder_asset,tokens);model=fresh_adapter(encoder,seed);cache_key=fingerprint({'text':text.key,'image_pool':images.pool.key})
    batch_fn=lambda rows,training:batch_for(rows,kind,images,tokens,text,training)
    with measured(folder,kind+'_pretraining_validation'):
        before=tensor_hash(model);vision=immutable_backbone(base);frozen=tensor_hash(encoder) if encoder is not None else None
        probe=smoke_model(model,batch_fn(data['train'].iloc[:CFG[kind]['batch_size']],True),kind,CFG[kind])
        require(tensor_hash(model)==before and immutable_backbone(base)==vision,'Preflight changed main initialization/vision')
        if encoder is not None:require(tensor_hash(encoder)==frozen and all(not p.requires_grad and p.grad is None for p in encoder.parameters()),'Adapter preflight changed frozen encoder')
        write_json(folder/'test_report.json',{'passed':[probe],'failed':[],'skipped':[],'kind':kind,'seed':seed})
    budget=(CFG['encoder']['epochs'] if kind=='encoder' else CFG['adapter']['max_epochs']) if phase=='development' else lock[kind+'_epochs']
    binding=stage_binding(package,data,phase,seed,kind,cache_key,encoder_asset,lock,budget)
    eval_fn=(lambda m:retrieval_evaluate(m,data,images,batch_fn,kind,top_k=0)[:2]) if phase=='development' else None
    with measured(folder,kind+'_training_validation_checkpoint'):
        done=optimize_stage(model,data['train'],batch_fn,eval_fn,folder,LOCAL/phase/f'seed_{seed}'/kind,binding,CFG[kind],budget,CFG['adapter']['patience'] if phase=='development' and kind=='adapter' else None)
    require(immutable_backbone(base)==vision,'Vision backbone changed during training')
    if encoder is not None:require(tensor_hash(encoder)==frozen and all(not p.requires_grad and p.grad is None for p in encoder.parameters()),'Frozen encoder/projection changed during adapter training')
    last=asset_record(folder/'last.pt',phase,kind,seed,done['epoch']);asset=asset_record(folder/('best.pt' if phase=='development' else 'final.pt'),phase,kind,seed,done['best_epoch'] if phase=='development' else budget,encoder_sha=encoder_asset['sha256'] if encoder_asset else None)
    receipts={str(folder/'training_history.csv'):file_hash(folder/'training_history.csv'),str(folder/'metric_history.json'):file_hash(folder/'metric_history.json')};metric=None;history=[]
    if phase=='development':
        selected=validate_asset(asset);model.load_state_dict(selected['model'],strict=True);del selected
        with measured(folder,'selected_checkpoint_full_evaluator'):
            metric,outputs=export_evaluation(folder,model,data,images,batch_fn,kind);receipts.update({str(folder/k):v for k,v in outputs.items()})
        history,epochfiles=verify_epoch_history(folder,done['metrics'],data);receipts.update(epochfiles);metric_equal(metric,history[done['best_epoch']-1]['metrics'])
    record={'schema':SCHEMA,'science_sha256':SCIENCE_SHA256,'phase':phase,'kind':kind,'seed':seed,'data':data_identity(data),'recipe':encoder_recipe() if kind=='encoder' else adapter_recipe(),'encoder_asset':encoder_asset,'lock_sha256':fingerprint(lock) if lock else None,'asset':asset,'last':last,'finished_epochs':done['epoch'],'budget_epochs':budget,'history':history,'metrics':metric,'file_receipts':receipts,'timing':done['timing'],'initial_sha256':done['initial_sha256'],'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),'origin':binding['origin']}
    branch_complete(folder,data,phase,seed,kind,encoder_asset,lock,_candidate=record);write_json(folder/'result.json',record,immutable=True)
    model=None;encoder=None;text=None;return record

def development_seed(package,seed,base,tokenizer,images):
    if seed==42:return import_seed42(package)
    path=ROOT/'development'/f'seed_{seed}'/'result.json'
    if path.exists():return verify_development_record(read_json(path),package)
    encoder=train_branch(package,'development',seed,'encoder',base,tokenizer,images)
    adapter=train_branch(package,'development',seed,'adapter',base,tokenizer,images,encoder['asset'])
    receipts={**encoder['file_receipts'],**adapter['file_receipts']}
    record={'schema':SCHEMA,'phase':'development','seed':seed,'semantic':semantic_development(package,seed),'encoder':encoder['asset'],'adapter':adapter['asset'],'encoder_last':encoder['last'],'adapter_last':adapter['last'],'encoder_history':encoder['history'],'adapter_history':adapter['history'],'encoder_metrics':encoder['metrics'],'adapter_metrics':adapter['metrics'],'adapter_best_epoch':adapter['asset']['epochs'],'per_query_path':str(stage_folder('development',seed,'adapter')/'per_query_results.csv'),'topk_path':str(stage_folder('development',seed,'adapter')/'top_k_results.csv'),'file_receipts':receipts,'timing':{'encoder':encoder['timing'],'adapter':adapter['timing'],'definition':'Encoder and adapter compute/wall/validation/checkpoint kept separate; inclusive cache/load/stage wall ledgers also retained, never claim whole-pipeline cost from adapter compute only'}}
    verify_development_record(record,package);write_json(path,record,immutable=True);return record

def select_common_epoch(records,seeds):
    require(set(records)==set(seeds),'All three completed development seeds required before final lock')
    curves={s:{x['epoch']:x['metrics']['Mean'] for x in records[s]['adapter_history']} for s in seeds}
    for s in seeds:
        require(records[s]['phase']=='development' and records[s]['seed']==s and records[s]['semantic']['seed_scope']=='full_pipeline' and records[s]['semantic']['encoder_seed']==records[s]['semantic']['adapter_seed']==s,'Not a complete full-pipeline development seed')
        require(all(math.isfinite(v) for v in curves[s].values()) and curves[s],'Missing/nonfinite actual epoch metrics')
    common=sorted(set.intersection(*(set(curves[s]) for s in seeds)));require(common,'No epoch with real metrics in all seeds; final training remains locked')
    means={e:float(np.mean([curves[s][e] for s in seeds])) for e in common};best=max(means.values());selected=next(e for e in common if means[e]==best)
    return {'eligible_epochs':common,'mean_development_Mean_percent':{str(e):means[e] for e in common},'selected_epoch':selected,'rule':'Earliest exact tie, maximize mean Dev Mean only over epochs present in all seeds; no imputation/interpolation','official_val_used':False}

def collect_development(package):
    records={};missing=[]
    for seed in CFG['seeds']:
        path=ROOT/'development'/f'seed_{seed}'/'result.json'
        if not path.exists():missing.append(seed);continue
        records[seed]=verify_development_record(read_json(path),package)
    return records,missing

def expected_final_lock(package,records):
    selection=select_common_epoch(records,CFG['seeds'])
    sources={str(s):{'encoder':records[s]['encoder']['sha256'],'adapter':records[s]['adapter']['sha256'],'epoch_history_sha256':fingerprint(records[s]['adapter_history']),'file_receipts':records[s]['file_receipts']} for s in CFG['seeds']}
    return {'schema':SCHEMA,'science_sha256':SCIENCE_SHA256,'seeds':CFG['seeds'],'seed_scope':'full_pipeline','encoder_recipe':encoder_recipe(),'adapter_recipe':adapter_recipe(),'encoder_epochs':3,'encoder_budget_note':'Predeclared common experimental budget; not claimed optimal','adapter_epochs':selection['selected_epoch'],'selection':selection,'development_data':data_identity(package.data('development')),'final_train_sha256':package.hashes['official_train_queries'],'protocol':package.binding,'feature_contract':package.contract,'development_sources':sources,'official_val_metric_used':False}

def lock_final_training(package):
    require(CFG['enable_lock'] is True,'Lock creation disabled; explicitly enable_lock=True after reviewing development results')
    records,missing=collect_development(package);require(not missing,'Final locked: missing completed development seeds '+str(missing));lock=expected_final_lock(package,records)
    write_json(ROOT/'final_training_lock.json',lock,immutable=True);write_json(ROOT/'lock_status.json',{'ready':True,'lock_sha256':fingerprint(lock),'selected_adapter_epoch':lock['adapter_epochs'],'official_val_metric_used':False});return lock

def read_final_lock(package):
    path=ROOT/'final_training_lock.json';require(path.is_file(),'Final training locked: complete all three development seeds and run phase lock')
    lock=read_json(path);records,missing=collect_development(package);require(not missing,'Locked development source now missing')
    require(lock==expected_final_lock(package,records),'Final lock/development science changed; preserve existing lock/results and stop')
    return lock

def final_seed(package,seed,base,tokenizer,images):
    lock=read_final_lock(package);encoder=train_branch(package,'final',seed,'encoder',base,tokenizer,images,lock=lock)
    adapter=train_branch(package,'final',seed,'adapter',base,tokenizer,images,encoder['asset'],lock)
    write_json(ROOT/'final'/f'seed_{seed}'/'training_complete.json',{'seed':seed,'encoder':encoder['asset'],'adapter':adapter['asset'],'final_lock_sha256':fingerprint(lock),'official_val_evaluated':False},immutable=True)
    return encoder,adapter


In [ ]:
# benchmark.py — embedded runtime
def verify_benchmark_result(folder,data,seed,encoder,adapter,lock,_candidate=None):
    path=Path(folder)/'result.json'
    if _candidate is None and not path.exists():return None
    r=read_json(path) if _candidate is None else _candidate;require(r['schema']==SCHEMA and r['phase']=='benchmark' and r['seed']==seed and r['encoder_asset']==encoder and r['adapter_asset']==adapter and r['lock_sha256']==fingerprint(lock) and r['data']==data_identity(data),'Official-val result provenance mismatch')
    verify_receipts(r['file_receipts']);q=pd.read_csv(Path(folder)/'per_query_results.csv',dtype={'query_id':str});metric_equal(metrics_from_predictions(q,data),r['metrics']);verify_topk(pd.read_csv(Path(folder)/'top_k_results.csv',dtype={'query_id':str,'image_id':str}),q,data,CFG['export_top_k'])
    require(encoder['phase']=='final' and (adapter is None or adapter['phase']=='final') and r['metrics']['n_query']==len(data['validation']),'Development model/result mistaken for official-val')
    return r

def benchmark_seed(package,seed,base,tokenizer,images):
    require(CFG['enable_benchmark'] is True,'Official-val benchmark disabled; enable explicitly only after final training is complete')
    lock=read_final_lock(package);train_data=package.data('final')
    e=branch_complete(stage_folder('final',seed,'encoder'),train_data,'final',seed,'encoder',None,lock);require(e is not None,'Final encoder training incomplete')
    a=branch_complete(stage_folder('final',seed,'adapter'),train_data,'final',seed,'adapter',e['asset'],lock);require(a is not None,'Final adapter training incomplete')
    data=package.data('benchmark');folder=ROOT/'benchmark'/f'seed_{seed}';folder.mkdir(parents=True,exist_ok=True)
    for asset in (e['asset'],a['asset']):require(asset['phase']=='final' and asset['seed']==seed,'Final benchmark checkpoint phase/seed mismatch')
    existing={label:verify_benchmark_result(folder/label,data,seed,e['asset'],None if label=='conventional' else a['asset'],lock) for label in ('conventional','adapter')}
    if all(existing.values()):return existing
    with measured(folder,'paired_official_val_cache_and_evaluation'):
        prepare_inputs(data,images,folder);tokens=Tokens(data['validation'],tokenizer);encoder=load_encoder_asset(e['asset'],base,tokenizer).freeze();text=TextFeatures(folder,data,encoder,e['asset'],tokens)
        fn=lambda rows,training:batch_for(rows,'adapter',images,tokens,text,False)
        for label in ('conventional','adapter'):
            if existing[label]:continue
            selected=folder/label;selected.mkdir(parents=True,exist_ok=True)
            model=CachedConventional(encoder.projection.weight).to(DEVICE) if label=='conventional' else load_adapter_asset(a['asset'],encoder)
            metric,outputs=export_evaluation(selected,model,data,images,fn,'adapter')
            result={'schema':SCHEMA,'phase':'benchmark','label':label,'seed':seed,'encoder_asset':e['asset'],'adapter_asset':None if label=='conventional' else a['asset'],'lock_sha256':fingerprint(lock),'data':data_identity(data),'metrics':metric,'file_receipts':{str(selected/k):v for k,v in outputs.items()},'paired_same_encoder':True,'seed_scope':'full_pipeline','selected_by_official_val':False}
            existing[label]=verify_benchmark_result(selected,data,seed,e['asset'],result['adapter_asset'],lock,_candidate=result);write_json(selected/'result.json',result,immutable=True);del model
        encoder=None;text=None
    return existing

def development_summary(package):
    records,missing=collect_development(package);rows=[]
    for seed in CFG['seeds']:
        r=records.get(seed)
        rows.append({'seed':seed,'encoder_seed':seed,'adapter_seed':seed,'seed_scope':'full_pipeline','status':'verified_complete' if r else 'missing','encoder_Dev_Mean':r['encoder_metrics']['Mean'] if r else None,'adapter_Dev_Mean':r['adapter_metrics']['Mean'] if r else None,'adapter_best_epoch':r['adapter_best_epoch'] if r else None})
    selection=select_common_epoch(records,CFG['seeds']) if not missing else None
    summary={'rows':rows,'all_three_development_verified':not missing,'missing_seeds':missing,'common_epoch_selection':selection,'official_val_used':False,'time_scope':'Read phase_timing and separate encoder/adapter timing. Seed42 historical full wall time is partly unknown; no pipeline speed claim based only on adapter compute.'}
    write_csv(ROOT/'development_summary.csv',pd.DataFrame(rows));write_json(ROOT/'development_summary.json',summary);return summary

def aggregate_paired(rows,seeds):
    expected={(s,label) for s in seeds for label in ('conventional','adapter')};require(set(rows)<=expected,'Unexpected seed/model in main table')
    complete=set(rows)==expected;aggregates=[];keys=[f'R@{k}' for k in (1,5,10,50)]+['Mean']
    for label in ('conventional','adapter'):
        members=[rows[s,label]['metrics'] for s in seeds if (s,label) in rows]
        r={'model':label,'seeds_evaluated':len(members),'seeds_expected':len(seeds),'complete':complete,'std_ddof':1}
        for key in keys:
            values=[m['Mean'] if key=='Mean' else m['overall'][key] for m in members];r[key+'_mean']=float(np.mean(values)) if complete else None;r[key+'_sample_std']=float(np.std(values,ddof=1)) if complete else None
        aggregates.append(r)
    delta={'model':'paired adapter minus conventional','complete':complete,'std_ddof':1}
    for key in keys:
        values=[]
        if complete:
            for s in seeds:
                left,right=rows[s,'adapter'],rows[s,'conventional'];require(left['encoder_asset']==right['encoder_asset'] and left['seed']==right['seed']==s,'Paired comparison used different encoders')
                a,b=left['metrics'],right['metrics'];values.append(a['Mean']-b['Mean'] if key=='Mean' else a['overall'][key]-b['overall'][key])
        delta[key+'_mean']=float(np.mean(values)) if complete else None;delta[key+'_sample_std']=float(np.std(values,ddof=1)) if complete else None
    return {'complete':complete,'model_aggregates':aggregates,'paired_delta':delta,'uncertainty_scope':'Sample standard deviation over three full-pipeline training seeds; no best-seed selection, no claim of statistical significance; paired bootstrap not included'}

def official_summary(package):
    lock=read_final_lock(package);data=package.data('benchmark');rows={};per_seed=[]
    for seed in CFG['seeds']:
        train=package.data('final');enc=branch_complete(stage_folder('final',seed,'encoder'),train,'final',seed,'encoder',None,lock)
        ad=branch_complete(stage_folder('final',seed,'adapter'),train,'final',seed,'adapter',enc['asset'],lock) if enc else None
        for label in ('conventional','adapter'):
            folder=ROOT/'benchmark'/f'seed_{seed}'/label
            result=verify_benchmark_result(folder,data,seed,enc['asset'],None if label=='conventional' else ad['asset'],lock) if enc and ad else None
            if result:rows[seed,label]=result
            per_seed.append({'seed':seed,'model':label,'status':'evaluated' if result else 'not_evaluated','Mean':result['metrics']['Mean'] if result else None,**{f'R@{k}':result['metrics']['overall'][f'R@{k}'] if result else None for k in (1,5,10,50)}})
    summary=aggregate_paired(rows,CFG['seeds']);summary.update(per_seed=per_seed,scope='FashionIQ official validation, not official test',lock_sha256=fingerprint(lock),protocol=package.protocol,paired_encoder=True)
    write_csv(ROOT/'official_val_per_seed.csv',pd.DataFrame(per_seed));write_csv(ROOT/'official_val_mean_std.csv',pd.DataFrame(summary['model_aggregates']+[summary['paired_delta']]));write_json(ROOT/'official_val_summary.json',summary)
    text='Conventional-only + retrieval residual adapter. Official-val complete: '+str(summary['complete'])+'. All three seeds required for mean±sample std; no best-seed selection. See JSON/CSV for detail.\nCompared with prior XAA/DVR, this branch includes additional text-encoder training; do not attribute the entire difference to the residual adapter. Encoder/cache/adapter/validation/checkpoint time are separate in phase reports.\n'
    write_bytes(ROOT/'summary.md',text.encode());return summary

def require_campaign_preflight(package,base):
    require(_PREFLIGHT_RECEIPT is not None and _PREFLIGHT_RECEIPT['session']==SESSION_ID and _PREFLIGHT_RECEIPT['package_id']==id(package) and _PREFLIGHT_RECEIPT['base_id']==id(base),'Run preflight in this runtime first')
    require(_PREFLIGHT_RECEIPT['binding']==preflight_binding(package) and _PREFLIGHT_RECEIPT['vision']==immutable_backbone(base),'Config/data/code/vision changed after preflight')

def run_selected_phase(package,base,tokenizer,images):
    require_campaign_preflight(package,base);phase=CFG['phase'];results={}
    if phase=='preflight':print('Preflight only. No full training, lock creation or official-val benchmark executed.');return
    require(phase in ('development','lock','final','benchmark','summaries'),'Choose a valid phase')
    require(CFG['seeds_to_run'] and len(set(CFG['seeds_to_run']))==len(CFG['seeds_to_run']) and set(CFG['seeds_to_run'])<=set(CFG['seeds']),'Selected seed must be a unique nonempty subset of 42,43,44')
    if phase=='development':
        for seed in CFG['seeds_to_run']:results[seed]=development_seed(package,seed,base,tokenizer,images)
        return development_summary(package)
    if phase=='lock':return lock_final_training(package)
    if phase=='final':
        require(CFG['enable_training'] is True,'Final training disabled')
        for seed in CFG['seeds_to_run']:results[seed]=final_seed(package,seed,base,tokenizer,images)
        return {'trained_seeds':list(results),'official_val_evaluated':False}
    if phase=='benchmark':
        for seed in CFG['seeds_to_run']:results[seed]=benchmark_seed(package,seed,base,tokenizer,images)
        return official_summary(package)
    return {'development':development_summary(package),'official_val':official_summary(package) if (ROOT/'final_training_lock.json').exists() else None}


In [ ]:
# validation.py — embedded runtime
def frozen_text_for_smoke(encoder,modifier):
    # Trainable adapter layers must be able to save these frozen inputs for backward.
    with torch.no_grad():return encoder.encode_text(modifier)

def smoke_model(main,batch,kind,hp):
    before=tensor_hash(main);saved=rng_state();model=copy.deepcopy(main);model.train();params=[p for p in model.parameters() if p.requires_grad]
    try:
        model.eval()
        with torch.no_grad():
            q=query_forward(model,batch,kind);one={k:({j:v[:1] for j,v in x.items()} if isinstance(x,dict) else x[:1]) for k,x in batch.items()}
            require(query_forward(model,one,kind).shape==(1,q.shape[1]),'Batch1 shape invalid')
            changed={**batch,'target':batch['target'].flip(0)};require(torch.equal(q,query_forward(model,changed,kind)),'Target leaked into query')
            if kind=='adapter':
                baseline=CachedConventional(model.projection_weight).to(q.device);off=model(batch['reference_features'],batch['text_features'],batch['mask'],batch['eos'],gates_off=True)
                require(torch.allclose(off,query_forward(baseline,batch,'adapter'),atol=CFG['gate_equivalence_atol'],rtol=0),'Gate-off conventional equivalence failed')
                dirty={**batch,'text_features':batch['text_features'].clone()};dirty['text_features'][batch['mask']==0]=12345
                require(torch.equal(q,query_forward(model,dirty,'adapter')),'Padding changed query')
            else:
                native=unit(unit(model.projection(model.text(**batch['modifier']).pooler_output))+unit(batch['reference_features']['pooled']))
                require(torch.allclose(q,native,atol=CFG['native_atol'],rtol=CFG['native_rtol']),'Native conventional forward differs')
        opt=torch.optim.AdamW(params,lr=hp['learning_rate'],weight_decay=hp['weight_decay'],betas=(.9,.999),eps=1e-8);stages=[]
        for step in range(4):
            model.train();opt.zero_grad(set_to_none=True);teacher=batch['target'].detach().clone().requires_grad_();b={**batch,'target':teacher}
            loss=retrieval_loss(query_forward(model,b,kind),teacher,b['target_ids'],hp['retrieval_temperature']);loss.backward();require(teacher.grad is None and all(p.grad is not None for p in params),'Teacher attached or trainable parameter disconnected')
            grad=torch.nn.utils.clip_grad_norm_(params,hp['gradient_clip'],error_if_nonfinite=True)
            if kind=='adapter':
                norms={'gate':sum(float(p.grad.detach().square().sum().cpu()) for p in (model.gate_ri,model.gate_ta)), 'RI':sum(float(p.grad.detach().square().sum().cpu()) for n,p in model.named_parameters() if p.grad is not None and n.startswith(('reference_','text_down','cross_attention','ri_mlp'))),'TA':sum(float(p.grad.detach().square().sum().cpu()) for n,p in model.named_parameters() if p.grad is not None and n.startswith('ta_mlp'))}
                if step==0:require(norms['RI']==norms['TA']==0,'Expected zero-gated branch gradient at initial step')
                stages.append(norms)
            opt.step();finite(loss,'smoke loss');finite(grad,'smoke gradient')
        if kind=='adapter':require(stages[-1]['RI']>0 and stages[-1]['TA']>0,'RI/TA still disconnected after gate updates')
        with tempfile.TemporaryDirectory(prefix='full_experiment_smoke_') as tmp:
            p=Path(tmp)/'smoke.pt';sig=fingerprint({'smoke':kind});state={'signature':sig,'phase':'smoke','step':4,'model':cpu_tree(model.state_dict()),'optimizer':cpu_tree(opt.state_dict()),'rng':rng_state()};checkpoint_save(p,state,sig)
            def update():
                opt.zero_grad(set_to_none=True);loss=retrieval_loss(query_forward(model,batch,kind),batch['target'],batch['target_ids'],hp['retrieval_temperature']);loss.backward();opt.step()
            update();expected=cpu_tree(model.state_dict());expected_rng=rng_state();saved_state=checkpoint_load(p,sig);model.load_state_dict(saved_state['model'],strict=True);opt.load_state_dict(saved_state['optimizer']);restore_rng(saved_state['rng']);update()
            require(all(torch.equal(v,model.state_dict()[k].detach().cpu()) for k,v in expected.items()) and same_rng(expected_rng,rng_state()),'Optimizer/model/RNG resume differs')
        return {'passed':True,'kind':kind,'optimizer_steps':4,'gate_off_equivalence':kind=='adapter','native_parity':kind=='encoder','batch1_and_normal':True,'target_independent_query':True,'teacher_detached':True,'branch_gradient_stages':stages,'save_load_resume':True,'main_model_unchanged':True,'main_RNG_restored':True}
    finally:
        restore_rng(saved);require(tensor_hash(main)==before,'Smoke modified main initialization');model=None

def fixture_tests():
    with isolated_rng(),tempfile.TemporaryDirectory(prefix='full_experiment_storage_') as tmp:
        root=Path(tmp);specs={'x':{'ids':['a','b'],'shape':[3],'dtype':'float32'}};s=TrialStore(root/'drive',root/'local',{'fixture':'v1'},specs,2);s.prepare('x',lambda ids:np.ones((len(ids),3),np.float32))
        marker=s.root/'x'/'0000000_0000002.json';raw=marker.read_bytes();m=read_json(marker);m['fingerprint']='wrong';write_json(marker,m)
        try:s.prepare('x',lambda ids:None)
        except RuntimeError:pass
        else:raise AssertionError('Wrong committed fingerprint accepted')
        marker.write_bytes(raw);p=marker.with_suffix('.npy');data=p.read_bytes();p.write_bytes(data[:-2]+b'xx')
        try:s.prepare('x',lambda ids:None)
        except RuntimeError:pass
        else:raise AssertionError('Corrupt cache accepted')
        p.write_bytes(data);pending=marker.with_suffix('.pending.json');pending.write_bytes(raw);marker.unlink();s.prepare('x',lambda ids:None)
        require(s.take('x',['b','a']).shape==(2,3),'ID cache mapping')
        q=np.array([[1,0]],np.float32);g=np.array([[1,0],[1,0]],np.float32);require(exact_ranks(q,g,[1]).tolist()==[2],'Stable tie ranking')
        require(assess_captions(['','long'])==(None,'single_empty_caption') and caption_text('','long')==' and long' and assess_captions(['.?, ',' '])[0]=='both_empty_captions','Caption policy/schema invalid')
        per={c:{'R@1':1.,'R@5':5.,'R@10':10.+i*10,'R@50':60.+i*10} for i,c in enumerate(CATEGORIES)};require(macro_metrics(per)['Mean']==45.,'Macro Mean arithmetic')
        require(positive_mask(['a','a','b'],'cpu').tolist()==[[True,True,False],[True,True,False],[False,False,True]],'Duplicate target positives')
        curves={seed:{'phase':'development','seed':seed,'semantic':{'seed_scope':'full_pipeline','encoder_seed':seed,'adapter_seed':seed},'adapter_history':[{'epoch':e,'metrics':{'Mean':v}} for e,v in values]} for seed,values in {42:[(1,10),(2,20),(3,30)],43:[(1,10),(2,20)],44:[(1,10),(2,20),(3,40)]}.items()}
        selection=select_common_epoch(curves,[42,43,44]);require(selection['eligible_epochs']==[1,2] and selection['selected_epoch']==2,'Missing epoch interpolated/ignored')
        for r in curves.values():r['adapter_history'][1]['metrics']['Mean']=10
        require(select_common_epoch(curves,[42,43,44])['selected_epoch']==1,'Earliest tie not selected')
        try:select_common_epoch({42:curves[42]},[42,43,44])
        except RuntimeError:pass
        else:raise AssertionError('Lock allowed missing seeds')
    return {'cache_committed_marker_corruption_fingerprint_pending':True,'caption_cases':True,'duplicate_target_loss_mask':True,'ranking_tie_Mean':True,'common_real_epoch_intersection_and_earliest_tie':True,'missing_seed_lock_blocked':True}

def preflight_binding(package):
    controls={'phase','enable_training','enable_lock','enable_benchmark','seeds_to_run'}
    return fingerprint({'config':{k:v for k,v in CFG.items() if k not in controls},'package':package.binding,'core_frame_shas':{name:frame_sha(package.frames[name]) for name in ('development_train','development_val','development_gallery','official_train_queries')},'code':CODE_SHA256,'science':SCIENCE_SHA256,'runtime':physical_runtime()})

def campaign_preflight(package,base,tokenizer,processor,images):
    global _PREFLIGHT_RECEIPT
    _PREFLIGHT_RECEIPT=None;saved=rng_state();report={'passed':[],'failed':[],'skipped':[],'ready_for_selected_phase':False,'session':SESSION_ID,'scope':'Actual current runtime preflight; no main training'}
    try:
        require(BOOTSTRAP['passed'] and not BOOTSTRAP['dependency_scope']['critical_conflicts'],'Environment/bootstrap conflicts remain')
        require(DEVICE.type=='cuda','GPU preflight required; local CPU tests do not mark Colab ready')
        import psutil
        memory=psutil.virtual_memory();free,total=torch.cuda.mem_get_info();disk=shutil.disk_usage(LOCAL)
        require(memory.available>2*1024**3 and disk.free>8*1024**3 and free>2*1024**3,'Insufficient RAM/disk/VRAM before real preflight')
        report['resources']={'RAM_available':memory.available,'disk_free':disk.free,'CUDA_free':free,'CUDA_total':total}
        from transformers.generation import GenerationMixin
        require(isinstance(GenerationMixin,type),'GenerationMixin import invalid')
        require(base.config.vision_config.hidden_size==768 and base.config.projection_dim==512 and base.config.text_config.hidden_size==512,'Fashion-CLIP dimensions differ')
        report['passed'].append(fixture_tests());vision=immutable_backbone(base)
        mod=tokenizer(['Red and longer',' and blue'],padding='max_length',max_length=77,return_tensors='pt');mod={k:v.to(DEVICE) for k,v in mod.items()}
        pixels=processor(images=Image.fromarray(np.arange(224*224*3,dtype=np.uint32).reshape(224,224,3).astype(np.uint8)),return_tensors='pt')['pixel_values'].to(DEVICE)
        with torch.inference_mode():
            v=base.vision_model(pixel_values=pixels);require(v.last_hidden_state.shape==(1,50,768) and v.last_hidden_state.dtype==torch.float32,'Vision shape/dtype differs');finite(v.last_hidden_state,'pretrained vision input forward')
        record=import_seed42(package);data=package.data('development');folder=ROOT/'preflight';folder.mkdir(exist_ok=True);prepare_inputs(data,images,folder)
        tokens=Tokens(pd.concat([data['train'],data['validation']],ignore_index=True),tokenizer);encoder=load_encoder_asset(record['encoder'],base,tokenizer).freeze()
        imported_adapter=load_adapter_asset(record['adapter'],encoder);del imported_adapter
        batch_fn=lambda rows,training:batch_for(rows,'encoder',images,tokens,None,training)
        metric,q,top=retrieval_evaluate(encoder,data,images,batch_fn,'encoder',CFG['export_top_k'])
        deviations=[abs(metric['Mean']-LEGACY['encoder_metrics']['Mean'])]+[abs(metric['per_category'][c][f'R@{k}']-LEGACY['encoder_metrics']['per_category'][c][f'R@{k}']) for c in CATEGORIES for k in (1,5,10,50)]
        write_csv(folder/'frozen_baseline_per_query.csv',q);write_json(folder/'baseline_verification.json',{'passed':max(deviations)<=CFG['baseline_tolerance_pp'],'metrics':metric,'deviation_max_pp':max(deviations),'tolerance_pp':CFG['baseline_tolerance_pp'],'encoder_checkpoint_sha256':record['encoder']['sha256']})
        require(max(deviations)<=CFG['baseline_tolerance_pp'],'Frozen baseline differs: diagnose encoder/projection/captions/mapping before training; do not raise tolerance')
        rows=data['train'].iloc[:32];b=batch_fn(rows,True);temporary=copy.deepcopy(encoder);temporary.requires_grad_(True);temporary.text.embeddings.requires_grad_(False)
        report['passed'].append(smoke_model(temporary,{k:({n:v[:4] for n,v in x.items()} if isinstance(x,dict) else x[:4]) for k,x in b.items()},'encoder',CFG['encoder']));del temporary
        # Small real-cache adapter fixture, not a full cache or a full-data training run.
        h,m,e=frozen_text_for_smoke(encoder,tokens.take(rows.query_id.tolist()))
        refids=rows.reference_image_id.tolist();adapter_batch={'query_ids':rows.query_id.tolist(),'reference_features':{'tokens':torch.from_numpy(images.take('reference',refids)).to(DEVICE),'pooled':torch.from_numpy(images.take('reference_pooled',refids)).to(DEVICE)},'text_features':h,'mask':m,'eos':e,'target':b['target'],'target_ids':b['target_ids']}
        adapter=fresh_adapter(encoder,42);require(tensor_hash(adapter)==LEGACY['adapter_initial_sha256'],'Seed42 fresh adapter initialization differs from verified source')
        report['passed'].append(smoke_model(adapter,adapter_batch,'adapter',CFG['adapter']));del adapter
        require(immutable_backbone(base)==vision and all(not p.requires_grad and p.grad is None for p in encoder.parameters()),'Frozen backbone/encoder changed')
        report['passed'].append({'GenerationMixin_CLIP_imports':True,'real_vision_input':True,'strict_legacy_encoder_adapter_load':True,'seed42_Dev_Mean':metric['Mean'],'train':16204,'val':1796,'full_pipeline_seed42_reuse_checked':True,'frozen_encoder_vision_unchanged':True})
        del encoder
        require(same_rng(saved,rng_state()),'Preflight changed main RNG')
        _PREFLIGHT_RECEIPT={'session':SESSION_ID,'package_id':id(package),'base_id':id(base),'binding':preflight_binding(package),'vision':vision};report['ready_for_selected_phase']=True
    except BaseException:report['failed'].append(traceback.format_exc());_PREFLIGHT_RECEIPT=None;raise
    finally:restore_rng(saved);write_json(ROOT/'test_report.json',report)
    print('Preflight ready; main training executed: False. Baseline:',metric['Mean']);return report


## 3. Preflight — bắt buộc, không full training
Đọc nguồn seed42 và strict-load checkpoint; tái hiện frozen baseline đủ1796 query. Kiểm real image forward, native conventional parity, gate-off, target detach, gradient/gate, optimizer step/save-load-resume bằng model tạm; giữ nguyên model/RNG chính. Cache source/full manifest được kiểm trước từng phase. Seed42 receipt là import read-only, không warm-start seed43/44/final. Local test không thay thế cell này trên GPU.


In [ ]:
init_campaign()
with measured(ROOT,'preparation_and_pretrained_load'):
    PACKAGE=Preparation(CFG['preparation_root'])
    BASE,TOKENIZER,PROCESSOR=load_pretrained(PACKAGE)
    IMAGES=Images(PACKAGE,BASE)
with measured(ROOT,'campaign_preflight'):
    PREFLIGHT=campaign_preflight(PACKAGE,BASE,TOKENIZER,PROCESSOR,IMAGES)


## 4. Chạy phase/seed và resume
- **A**: `phase='development'`, `seeds_to_run=[42,43,44]`, bật `enable_training=True`. Seed42 chỉ import nếu mọi artifact hợp lệ;43/44 train encoder riêng3epoch rồi adapter mới max10/patience3. Có thể chia seed sang nhiều phiên, cùng campaign.
- **B**: `phase='lock'`, bật `enable_lock=True` khi đủ ba result; chỉ dùng epoch giao thật của ba curve, mean Dev Mean và earliest tie. Thiếu seed/metric thì không ghi lock.
- **C**: `phase='final'`, chọn seed, bật `enable_training=True`. Khởi tạo encoder/adapter mới, train18000 theo epoch chung đã khóa; không eval/early-stop official-val, không warm-start development.
- **D**: `phase='benchmark'`, bật `enable_benchmark=True` sau final. So conventional-off với adapter trên chính encoder final từng seed; mean±sample std chỉ có khi đủ3 seed, không chọn best seed.

Có thể sửa các trường điều khiển trong CFG bằng cell riêng rồi chạy cell bên dưới; không cần reinit khi chỉ đổi phase/cờ/seed chọn. Đổi recipe/source/runtime cần preflight lại. Resume chỉ dùng last của đúng phase/seed/config/runtime/code; best dùng chọn development, final dùng benchmark. Kết quả complete chỉ skip sau checksum/checkpoint/coverage/metric/readback; lỗi không reset hoặc xóa run. Nếu active resume khác code/config thì dừng để review. Nguồn complete hợp đồng khoa học vẫn khớp được reuse qua semantic receipt, không train lại chỉ vì version/logging đổi.

Drive quota được kiểm trước training/cache lớn (Colab có thể yêu cầu xác thực Google để đọc storageQuota). Giữ batch/FP32 cố định; nếu thiếu disk/RAM/VRAM/quota thì dừng, không tự giảm cấu hình. Từng phase/seed lưu checkpoint, history, query/TopK CSV, cache/timing/resource/test JSON. Chi phí encoder được giữ riêng cùng cache/adapter/validation/checkpoint; timing lịch sử không đủ thì để unknown.


In [ ]:
with measured(ROOT,CFG['phase']+'_selected_phase_total'):
    RESULT=run_selected_phase(PACKAGE,BASE,TOKENIZER,IMAGES)
print('Selected phase:',CFG['phase'],'| full pipeline seeds:',CFG['seeds'],'| outputs:',ROOT)


In [ ]:
# Opt in explicitly to run all phases; defaults to preflight only.
RUN_FULL_PIPELINE = False
if RUN_FULL_PIPELINE:
    # Explicit sequential orchestration authorized by the user; no config-cell rerun.
    require(PREFLIGHT['ready_for_selected_phase'] and not PREFLIGHT['failed'], 'Mandatory preflight must pass')
    require_campaign_preflight(PACKAGE,BASE)
    LIVE_JOB = stamp().replace(':','-')+'_'+uuid.uuid4().hex[:8]
    LIVE_FOLDER = ROOT/'live_orchestration'/LIVE_JOB
    LIVE_FOLDER.mkdir(parents=True,exist_ok=False)
    LIVE_STATE = {'job':LIVE_JOB,'notebook':'https://colab.research.google.com/drive/1yioJceWiZfPUmYF396S7mzpd6o7gMEv2','seeds':[42,43,44],'status':'running','phase':'preflight','completed_phases':['preflight'],'preflight_report_sha256':file_hash(ROOT/'test_report.json'),'implementation_sha256':CODE_SHA256,'science_sha256':SCIENCE_SHA256,'output_root':str(ROOT),'started_utc':stamp()}
    write_json(LIVE_FOLDER/'state.json',LIVE_STATE)
    try:
        for live_phase in ('development','lock','final','benchmark'):
            CFG.update(phase=live_phase,seeds_to_run=[42,43,44],enable_training=live_phase in ('development','final'),enable_lock=live_phase=='lock',enable_benchmark=live_phase=='benchmark')
            LIVE_STATE.update(phase=live_phase,status='running',phase_started_utc=stamp())
            write_json(LIVE_FOLDER/'state.json',LIVE_STATE)
            print('START PHASE',live_phase,'| full-pipeline seeds:',CFG['seeds_to_run'],flush=True)
            with measured(ROOT,'live_'+live_phase+'_inclusive'):
                LIVE_RESULT = run_selected_phase(PACKAGE,BASE,TOKENIZER,IMAGES)
            if live_phase=='development':
                LIVE_DEV,LIVE_MISSING = collect_development(PACKAGE)
                require(not LIVE_MISSING and set(LIVE_DEV)=={42,43,44},'Missing verified full-pipeline development seed')
                require(all(r['semantic']['encoder_seed']==r['semantic']['adapter_seed']==s for s,r in LIVE_DEV.items()),'Encoder/adapter seed mismatch')
                print('VERIFIED DEVELOPMENT:',[{k:row[k] for k in ('seed','encoder_Dev_Mean','adapter_Dev_Mean','adapter_best_epoch')} for row in LIVE_RESULT['rows']],flush=True)
            elif live_phase=='lock':
                LIVE_LOCK = read_final_lock(PACKAGE)
                LIVE_DEV,LIVE_MISSING = collect_development(PACKAGE)
                require(not LIVE_MISSING and LIVE_LOCK['selection']==select_common_epoch(LIVE_DEV,[42,43,44]),'Common actual-epoch selection mismatch')
                require(LIVE_LOCK['encoder_epochs']==3 and LIVE_LOCK['official_val_metric_used'] is False,'Final lock budget/protocol mismatch')
                print('VERIFIED LOCK: encoder epochs',LIVE_LOCK['encoder_epochs'],'| adapter epochs',LIVE_LOCK['adapter_epochs'],'| eligible epochs',LIVE_LOCK['selection']['eligible_epochs'],flush=True)
            elif live_phase=='final':
                LIVE_LOCK = read_final_lock(PACKAGE)
                LIVE_DATA = PACKAGE.data('final')
                require(len(LIVE_DATA['train'])==18000 and LIVE_DATA['validation'] is None,'Final training split mismatch')
                for live_seed in (42,43,44):
                    live_e = branch_complete(stage_folder('final',live_seed,'encoder'),LIVE_DATA,'final',live_seed,'encoder',None,LIVE_LOCK)
                    require(live_e is not None,'Final encoder incomplete: '+str(live_seed))
                    live_a = branch_complete(stage_folder('final',live_seed,'adapter'),LIVE_DATA,'final',live_seed,'adapter',live_e['asset'],LIVE_LOCK)
                    require(live_a is not None,'Final adapter incomplete: '+str(live_seed))
                print('VERIFIED FINAL: fresh encoder+adapter for seeds42/43/44, 18000 train, no official-val selection.',flush=True)
            else:
                LIVE_SUMMARY = official_summary(PACKAGE)
                require(LIVE_SUMMARY['complete'] and len(LIVE_SUMMARY['per_seed'])==6 and all(x['status']=='evaluated' for x in LIVE_SUMMARY['per_seed']),'Official-val result incomplete')
                require(len(PACKAGE.data('benchmark')['validation'])==6016,'Official-val query count mismatch')
                print('VERIFIED OFFICIAL-VAL:',LIVE_SUMMARY['per_seed'],flush=True)
                print('MEAN / SAMPLE STD:',LIVE_SUMMARY['model_aggregates'],'| PAIRED DELTA:',LIVE_SUMMARY['paired_delta'],flush=True)
            LIVE_STATE['completed_phases'].append(live_phase)
            LIVE_STATE.update(status='phase_verified',phase_finished_utc=stamp())
            write_json(LIVE_FOLDER/'state.json',LIVE_STATE)
            print('PHASE VERIFIED',live_phase,'| artifacts:',ROOT,flush=True)
        LIVE_STATE.update(status='complete',finished_utc=stamp())
        write_json(LIVE_FOLDER/'state.json',LIVE_STATE)
        print('FULL PIPELINE VERIFIED COMPLETE | job:',LIVE_JOB,'| outputs:',ROOT,flush=True)
    except BaseException as live_error:
        LIVE_STATE.update(status='blocked',blocked_utc=stamp(),error_type=type(live_error).__name__,error_message=str(live_error),stage_progress=[])
        for live_seed in (42,43,44):
            for live_kind in ('encoder','adapter'):
                live_status = stage_folder(CFG['phase'],live_seed,live_kind)/'status.json'
                if live_status.is_file():
                    live_progress = read_json(live_status)
                    LIVE_STATE['stage_progress'].append({k:live_progress.get(k) for k in ('phase','seed','kind','epoch','cursor','step','finished','stop_reason')})
        write_json(LIVE_FOLDER/'state.json',LIVE_STATE)
        write_bytes(LIVE_FOLDER/'traceback.txt',traceback.format_exc().encode())
        print('PIPELINE BLOCKED | phase:',CFG['phase'],'| progress:',LIVE_STATE['stage_progress'],'| preserved checkpoints and artifacts | log:',LIVE_FOLDER,flush=True)
        raise


## 5. Báo cáo
`development_summary.json/csv`, `final_training_lock.json`, `official_val_summary.json`, `official_val_per_seed.csv`, `official_val_mean_std.csv`, `summary.md`. Metric chưa chạy đểnull. Kết quả official-val không phải official test. So với XAA/DVR cũ phải ghi rõ nhánh này có thêm giai đoạn train text encoder; không quy toàn bộ chênh lệch cho adapter. Sample std ba seed là độ biến thiên training, không phải kết luận significance; không kèm bootstrap.
